# 📘 Notebook Mestre de Análise de Dados
### Manual prático e template de referência

> **Recebi um novo dataset. Por onde começo?** Este notebook responde a essa pergunta seguindo a ordem real de um projeto: importar → conhecer → limpar → descrever → explorar → testar → comunicar.

Cada tópico segue o formato:

| Bloco | Conteúdo |
|---|---|
| **O que faz** | o objetivo da técnica |
| **Quando usar** | a situação em que ela se aplica |
| **Código** | template genérico, pronto para adaptar |
| **⚠️ O que observar** | armadilhas, interpretação e cuidados |

**Convenções do template**

- `df` é sempre o DataFrame de trabalho, e `df_original` é a cópia intocada dos dados brutos.
- Os nomes de colunas são genéricos e indicam o **tipo** da variável. Basta trocá-los pelos nomes reais:

| Nome no template | Representa |
|---|---|
| `'coluna_id'` | identificador da linha |
| `'coluna_num'`, `'coluna_num_2'` | variáveis numéricas (quantitativas) |
| `'coluna_cat'`, `'coluna_cat_2'` | variáveis categóricas nominais |
| `'coluna_ordinal'` | categórica com ordem (ex.: baixo < médio < alto) |
| `'coluna_binaria'` | variável 0/1 (sim/não) |
| `'coluna_data'`, `'coluna_data_fim'` | datas |
| `'coluna_valor_texto'` | número armazenado como texto (ex.: `"1.234,56"`) |
| `'coluna_multivalor'` | texto com vários valores separados por `;` |

- Duas tabelas auxiliares completam o exemplo e permitem praticar SQL, junções e métricas de negócio:

| Tabela | Colunas | Representa |
|---|---|---|
| `tabela_categorias` | `coluna_cat`, `coluna_grupo`, `coluna_meta` | tabela de dimensão (atributos de cada categoria) |
| `eventos` | `coluna_id_cliente`, `coluna_variante`, `coluna_data_evento`, `coluna_etapa`, `coluna_valor_pedido` | um registro por evento de cliente (etapas de funil e pedidos) |

- O notebook roda do início ao fim com um **dataset sintético** gerado na Seção 1, criado só para testar o template. Para usar dados reais, altere `USAR_DADOS_EXEMPLO = False` e informe o caminho do arquivo.
- 🔹 **Alternativa**: outra forma válida de fazer a mesma tarefa, mostrada só quando agrega valor.
- ➕ **Complemento**: técnica que não estava no material de estudo original, incluída porque fecha uma lacuna do fluxo de trabalho.

## Índice

| # | Seção | Pergunta que responde |
|---|---|---|
| 0 | Roteiro rápido | Qual a sequência de trabalho? |
| 1 | Configuração do ambiente | Quais bibliotecas e opções preparar? |
| 2 | Importação dos dados | Como carregar arquivos e consultar bancos de dados (SQL)? |
| 3 | Primeira inspeção | Qual a estrutura, que tipos de dados há e o que está faltando? |
| 4 | Limpeza e tratamento | Como deixar os dados prontos para análise e combinar tabelas? |
| 5 | Estatística descritiva | Como resumir cada variável em números? |
| 6 | Análise univariada (gráficos) | Qual o formato de cada distribuição? |
| 7 | Outliers | Existem valores extremos? O que fazer com eles? |
| 8 | Análise bivariada e multivariada | Como as variáveis se relacionam? |
| 9 | Análise temporal | Como os dados se comportam no tempo e quanto variam entre períodos? |
| 10 | Métricas de negócio | Como medir desempenho: KPIs, funil, coorte e retenção? |
| 11 | Visualização: referência técnica | Como construir, combinar e formatar gráficos? |
| 12 | Estatística inferencial | As diferenças observadas são estatisticamente significativas? E o teste A/B? |
| 13 | Regressão linear | Qual é a tendência linear entre duas variáveis, e quanto ela explica? |
| 14 | Insights e comunicação | Como transformar resultados em conclusões e entregáveis? |
| 15 | Checklist final | Esqueci alguma etapa? |

## 0. Roteiro rápido: recebi um novo dataset

A Análise Exploratória de Dados (**EDA**) é a etapa inicial de qualquer projeto. O objetivo é entender a "cara" dos dados antes de tirar conclusões. As perguntas que guiam a EDA são:

1. **Estrutura**: como os dados estão organizados? O que é cada linha e cada coluna?
2. **Tipos**: com que tipos de dados estou trabalhando? Os tipos lidos estão corretos?
3. **Qualidade**: há nulos, duplicados, valores inválidos ou formatos inconsistentes?
4. **Relevância**: quais colunas são úteis para a pergunta de negócio? Quais posso ignorar, e por quê?
5. **Insights**: que padrões, diferenças e relações os dados revelam?

**Sequência de trabalho**

```text
Pergunta de negócio
   └─► Importar / consultar SQL (Seção 2) ─► Inspecionar (3) ─► Limpar e combinar tabelas (4)
          └─► Descrever (5) ─► Visualizar distribuições (6) ─► Outliers (7)
                 └─► Relacionar variáveis (8) ─► Tempo e variações (9) ─► KPIs (10)
                        └─► Testar hipóteses e A/B (12, 13) ─► Interpretar e comunicar (14)
```

⚠️ **O que observar**
- A EDA é **iterativa**. Um gráfico na Seção 8 pode revelar um problema que exige voltar à limpeza (Seção 4).
- Tabelas não bastam. Muitos padrões, como valores arredondados, grupos ocultos e caudas longas, só aparecem quando os dados são **plotados de formas diferentes**.
- Registre cada decisão (o que removeu, filtrou ou transformou, e por quê). Isso será usado no detalhamento técnico do relatório (Seção 14).

---
# 1. Configuração do ambiente

## 1.1 Bibliotecas

**O que faz:** importa as bibliotecas usadas no fluxo de análise.

| Biblioteca | Papel |
|---|---|
| `pandas` | leitura, limpeza, transformação e agregação de tabelas |
| `numpy` | operações numéricas, arrays, funções matemáticas (`log`, `where`, ...) |
| `matplotlib` | base de todos os gráficos (figura, eixos, títulos, salvar) |
| `seaborn` | gráficos estatísticos de alto nível sobre o matplotlib |
| `scipy.stats` | estatística descritiva avançada, correlação, testes de hipóteses, regressão simples |
| `statistics` | variância e desvio padrão amostral/populacional explícitos |
| `statsmodels` | Tukey, teste Z, teste de proporções, poder e tamanho de amostra, regressão OLS completa |
| `sqlite3` *(biblioteca padrão)* | banco de dados SQL local para praticar e executar consultas |
| `pingouin` *(opcional)* | testes com tabela completa: efeito, IC, poder, Welch, Games-Howell |
| `scikit_posthocs` *(opcional)* | teste de Dunn (pós-teste não paramétrico) |

In [ ]:
# Instalação (rodar uma vez, apenas se faltar algum pacote)
# !pip install pandas numpy matplotlib seaborn scipy statsmodels pingouin scikit-posthocs openpyxl

In [ ]:
import statistics as st
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

%matplotlib inline

## 1.2 Opções de exibição e estilo dos gráficos

**Quando usar:** sempre no início. Isso evita que o pandas esconda colunas e padroniza a aparência dos gráficos.

⚠️ **O que observar**
- `display.max_rows = None` em tabelas grandes pode travar o notebook. Prefira um número (ex.: `100`).
- `float_format` muda apenas a **exibição**, não os valores armazenados.
- `sns.set_context` aumenta fontes e linhas para apresentações (`'notebook'` < `'talk'` < `'poster'`).

In [ ]:
pd.set_option('display.max_columns', None)           # mostra todas as colunas
pd.set_option('display.max_rows', 100)               # limite de linhas exibidas
pd.set_option('display.float_format', '{:,.2f}'.format)  # 2 casas decimais e separador de milhar

sns.set_theme(style='whitegrid', context='notebook', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)

## 1.3 Parâmetros do projeto

**O que faz:** centraliza caminhos e parâmetros em um único lugar.

**Quando usar:** quando a mesma análise é refeita periodicamente (por exemplo, todo mês ou todo ano). Basta alterar os parâmetros e rodar o notebook de novo.

⚠️ Evite caminhos absolutos da sua máquina espalhados pelo código. Prefira caminhos relativos à pasta do projeto.

In [ ]:
USAR_DADOS_EXEMPLO = True                # False -> usa o arquivo real abaixo

PERIODO = 'aaaa'                         # ex.: ano/mês de referência da análise
PASTA_DADOS = Path('dados')
PASTA_SAIDA = Path('output')

CAMINHO_DADOS = PASTA_DADOS / 'nome_do_arquivo.csv'
CAMINHO_TABELA_CATEGORIAS = PASTA_DADOS / 'nome_da_tabela_de_categorias.csv'
CAMINHO_EVENTOS = PASTA_DADOS / 'nome_da_tabela_de_eventos.csv'
CAMINHO_SAIDA_TABELA = PASTA_SAIDA / f'tabela_{PERIODO}.xlsx'
CAMINHO_SAIDA_GRAFICO = PASTA_SAIDA / f'grafico_{PERIODO}.png'

ALFA = 0.05                              # nível de significância dos testes (Seção 12)
SEED = 42                                # semente para resultados reprodutíveis

## 1.4 Datasets sintéticos de exemplo

**O que faz:** gera dados **fictícios e aleatórios** com os mesmos tipos de coluna do template, para que todas as células deste notebook possam ser executadas e testadas.

**Quando usar:** apenas para estudar ou testar o template. Em um projeto real, defina `USAR_DADOS_EXEMPLO = False`.

Os dados incluem, de propósito, alguns problemas típicos: nulos, duplicados, números em formato de texto, outliers e assimetria. Além da tabela principal (`df`), são geradas uma tabela de dimensão (`tabela_categorias`) e uma tabela de eventos de clientes (`eventos`), usadas em SQL, junções, métricas de negócio e testes A/B.

In [ ]:
def gerar_dados_exemplo(n=1000, seed=SEED):
    # Gera um DataFrame fictício com os tipos de coluna usados no template.
    rng = np.random.default_rng(seed)
    cat = rng.choice(['A', 'B', 'C'], size=n, p=[0.5, 0.3, 0.2])
    efeito_cat = pd.Series(cat).map({'A': 0.0, 'B': 0.25, 'C': 0.5}).to_numpy()

    num = np.round(rng.lognormal(mean=2.0 + efeito_cat, sigma=0.6, size=n), 1)  # assimétrica à direita
    num_2 = np.round(50 + 4 * num + rng.normal(0, 15, n), 2)                   # relação linear com num
    num_2[rng.choice(n, 8, replace=False)] *= 6                                # outliers artificiais

    binaria = (rng.random(n) < 1 / (1 + np.exp(-(num_2 - 160) / 40))).astype(int)
    data = pd.Timestamp('2024-01-01') + pd.to_timedelta(rng.integers(0, 365, n), unit='D')
    duracao = pd.to_timedelta(rng.integers(0, 15, n), unit='D')
    valores = rng.uniform(10, 5000, n)
    itens = ['item_1', 'item_2', 'item_3', 'item_4']

    df = pd.DataFrame({
        'coluna_id': np.arange(1, n + 1),
        'coluna_cat': cat,
        'coluna_cat_2': rng.choice(['X', 'Y'], size=n),
        'coluna_ordinal': rng.choice(['baixo', 'médio', 'alto'], size=n, p=[0.3, 0.45, 0.25]),
        'coluna_binaria': binaria,
        'coluna_num': num,
        'coluna_num_2': num_2,
        'coluna_valor_texto': [f'{v:,.2f}'.replace(',', '#').replace('.', ',').replace('#', '.') for v in valores],
        'coluna_data': data.strftime('%d/%m/%Y'),
        'coluna_data_fim': (data + duracao).strftime('%d/%m/%Y'),
        'coluna_multivalor': [';'.join(sorted(rng.choice(itens, rng.integers(1, 4), replace=False))) for _ in range(n)],
    })
    # Problemas típicos de dados reais
    df.loc[rng.choice(n, 50, replace=False), 'coluna_num'] = np.nan
    df.loc[rng.choice(n, 30, replace=False), 'coluna_cat'] = np.nan
    df.loc[rng.choice(n, 20, replace=False), 'coluna_multivalor'] = np.nan
    df = pd.concat([df, df.sample(5, random_state=seed)], ignore_index=True)  # linhas duplicadas
    return df

In [ ]:
def gerar_tabelas_auxiliares(n_clientes=3000, seed=SEED):
    # Gera uma tabela de dimensão por categoria e uma tabela de eventos de clientes (funil + pedidos).
    rng = np.random.default_rng(seed)

    # 'D' não existe em df e 'Não informado' não existe aqui: isso torna visíveis os tipos de junção (Seção 4.10)
    tabela_categorias = pd.DataFrame({
        'coluna_cat': ['A', 'B', 'C', 'D'],
        'coluna_grupo': ['grupo_1', 'grupo_1', 'grupo_2', 'grupo_2'],
        'coluna_meta': [1_200_000, 700_000, 450_000, 100_000],
    })

    fim = pd.Timestamp('2024-12-31')
    taxa_conversao = {'controle': 0.30, 'tratamento': 0.36}   # o 'tratamento' converte mais (teste A/B, Seção 12.8)
    linhas = []
    for cliente in range(1, n_clientes + 1):
        variante = str(rng.choice(['controle', 'tratamento']))
        data = pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(rng.integers(0, 700)))
        linhas.append((cliente, variante, data, 'etapa_1', np.nan))
        if rng.random() < 0.6:
            data += pd.Timedelta(days=int(rng.integers(0, 5)))
            linhas.append((cliente, variante, data, 'etapa_2', np.nan))
            if rng.random() < taxa_conversao[variante] / 0.6:
                data += pd.Timedelta(days=int(rng.integers(0, 10)))
                while data <= fim:                                   # pedidos recorrentes (retenção)
                    linhas.append((cliente, variante, data, 'etapa_3', round(rng.lognormal(5, 0.5), 2)))
                    if rng.random() > 0.65:
                        break
                    data += pd.Timedelta(days=int(rng.integers(25, 40)))

    eventos = pd.DataFrame(linhas, columns=['coluna_id_cliente', 'coluna_variante', 'coluna_data_evento',
                                            'coluna_etapa', 'coluna_valor_pedido'])
    return tabela_categorias, eventos

---
# 2. Importação dos dados

## 2.1 Leitura do arquivo

**O que faz:** carrega o arquivo em um DataFrame e guarda uma cópia dos dados brutos.

**Quando usar:** é sempre o primeiro passo. Mantenha `df_original` intocado para poder comparar ou recomeçar sem precisar reler o arquivo.

In [ ]:
if USAR_DADOS_EXEMPLO:
    df = gerar_dados_exemplo()
    tabela_categorias, eventos = gerar_tabelas_auxiliares()
else:
    df = pd.read_csv(CAMINHO_DADOS)
    tabela_categorias = pd.read_csv(CAMINHO_TABELA_CATEGORIAS)
    eventos = pd.read_csv(CAMINHO_EVENTOS, parse_dates=['coluna_data_evento'])

df_original = df.copy()   # cópia de segurança dos dados brutos
df.head()

### Parâmetros de leitura mais úteis

| Situação | Parâmetro | Exemplo |
|---|---|---|
| Arquivo separado por `;` (comum em dados brasileiros) | `sep` | `sep=';'` |
| Acentos aparecendo como `Ã§`, ou `UnicodeDecodeError` | `encoding` | `encoding='latin-1'` ou `'cp1252'` |
| Decimal com vírgula e milhar com ponto (`1.234,56`) | `decimal`, `thousands` | `decimal=',', thousands='.'` |
| Usar uma coluna como índice (ex.: ID) | `index_col` | `index_col='coluna_id'` |
| Ler só algumas colunas (arquivos grandes) | `usecols` | `usecols=['coluna_1', 'coluna_2']` |
| Converter datas já na leitura | `parse_dates`, `dayfirst` | `parse_dates=['coluna_data'], dayfirst=True` |
| Valores que significam "vazio" | `na_values` | `na_values=['-', 'NA', 'Não informado']` |

```python
# CSV no padrão brasileiro
df = pd.read_csv('nome_do_arquivo.csv', sep=';', encoding='cp1252', decimal=',', thousands='.')

# Excel (requer openpyxl)
df = pd.read_excel('nome_do_arquivo.xlsx', sheet_name='nome_da_aba')

# Vários arquivos com a mesma estrutura (ex.: um por mês/ano)
arquivos = sorted(Path('dados').glob('*_nome_do_arquivo.csv'))
df = pd.concat([pd.read_csv(a, sep=';') for a in arquivos], ignore_index=True)

# Datasets de prática embutidos no seaborn (lista: sns.get_dataset_names())
df_pratica = sns.load_dataset('nome_do_dataset')
```

⚠️ **O que observar**
- Se `df.info()` mostrar números como texto, quase sempre o problema está em `sep`, `decimal` ou `thousands`. Corrigir na leitura é mais limpo do que converter depois (Seção 4.2).
- Arquivos relacionados (por exemplo, uma tabela principal e tabelas de detalhe) podem ser combinados por uma chave em comum com `pd.merge` (Seção 4.10) ou diretamente em SQL (Seção 2.3).

## 2.2 Dicionário de dados

**O que faz:** associa cada coluna ao seu significado.

**Quando usar:** quando o dataset vem com um arquivo de descrição (schema). Ler esse arquivo antes de analisar evita interpretações erradas.

```python
dicionario = pd.read_csv('nome_do_dicionario.csv')

# Se as descrições vierem com tags HTML, remova-as com regex
dicionario['descricao'] = (dicionario['descricao']
                           .str.replace(r'<.*?>', '', regex=True)
                           .str.strip())

for nome, desc in zip(dicionario['nome_coluna'], dicionario['descricao']):
    print(f'{nome} -> {desc}')
```

⚠️ Sem dicionário, documente você mesmo o significado de cada coluna, a unidade de medida (R$, dias, %, ...) e a origem dos dados.

## 2.3 Consultas SQL ➕

**O que faz:** extrai, filtra, combina e agrega dados direto no banco com SQL e traz o resultado como DataFrame (`pd.read_sql_query`).

**Quando usar:** sempre que os dados estiverem em um banco de dados. Filtrar e agregar no banco é mais rápido e traz para o Python só o necessário.

Aqui, as tabelas do exemplo são carregadas em um banco **SQLite em memória** (biblioteca padrão do Python), o que permite executar as consultas sem servidor. Em um projeto real, troque apenas a conexão:

```python
conn = sqlite3.connect('nome_do_banco.db')                               # arquivo SQLite

from sqlalchemy import create_engine                                     # PostgreSQL, MySQL, SQL Server...
conn = create_engine(os.environ['URL_DO_BANCO'])                         # credenciais fora do código
df = pd.read_sql_query('SELECT ... FROM nome_da_tabela', conn)
```

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('tabela_principal', conn, index=False)
tabela_categorias.to_sql('tabela_categorias', conn, index=False)
eventos.to_sql('tabela_eventos', conn, index=False)


def consulta(sql, params=None):
    # Executa uma consulta SQL e devolve o resultado como DataFrame.
    return pd.read_sql_query(sql, conn, params=params)

### Selecionar, filtrar e ordenar

`SELECT` escolhe as colunas, `WHERE` filtra as linhas, `ORDER BY` ordena e `LIMIT` limita o resultado. Valores vindos de variáveis entram como **parâmetros** (`:limite`), nunca concatenados no texto da consulta.

In [ ]:
consulta('''
    SELECT coluna_id, coluna_cat, coluna_num_2
    FROM tabela_principal
    WHERE coluna_num_2 > :limite
      AND coluna_cat IN ('A', 'B')
    ORDER BY coluna_num_2 DESC
    LIMIT 10
''', params={'limite': 100})

### Agregar: `GROUP BY` e `HAVING`

`GROUP BY` agrupa as linhas e as funções de agregação (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) resumem cada grupo. `WHERE` filtra **antes** de agrupar, e `HAVING` filtra **depois**, usando o resultado da agregação.

In [ ]:
consulta('''
    SELECT coluna_cat,
           COUNT(*)                            AS n,
           COUNT(coluna_num)                   AS n_preenchidos,
           ROUND(AVG(coluna_num_2), 2)         AS media,
           SUM(coluna_binaria)                 AS n_binaria,
           ROUND(AVG(coluna_binaria) * 100, 1) AS pct_binaria
    FROM tabela_principal
    WHERE coluna_cat IS NOT NULL
    GROUP BY coluna_cat
    HAVING COUNT(*) >= 30
    ORDER BY n DESC
''')

### Combinar tabelas: `JOIN`

`LEFT JOIN` mantém todas as linhas da tabela da esquerda e traz da direita só o que tiver correspondência na chave; o que não tiver vira `NULL`. `INNER JOIN` mantém apenas as linhas com correspondência nas duas tabelas.

In [ ]:
consulta('''
    SELECT p.coluna_cat,
           c.coluna_grupo,
           c.coluna_meta,
           COUNT(*) AS n
    FROM tabela_principal AS p
    LEFT JOIN tabela_categorias AS c
           ON p.coluna_cat = c.coluna_cat
    GROUP BY p.coluna_cat, c.coluna_grupo, c.coluna_meta
    ORDER BY n DESC
''')

### Agregação condicional, CTE e funções de janela

- `CASE WHEN` cria regras dentro da consulta, por exemplo para contar só os registros que atendem a uma condição.
- `WITH` (CTE) nomeia uma consulta intermediária e deixa consultas longas legíveis.
- Funções de janela (`OVER`) calculam sobre um conjunto de linhas **sem agrupá-las**: acumulados (`SUM ... OVER`), valor do período anterior (`LAG`) e rankings (`RANK`, `ROW_NUMBER`).

In [ ]:
consulta('''
    SELECT coluna_variante,
           COUNT(DISTINCT coluna_id_cliente) AS clientes,
           COUNT(DISTINCT CASE WHEN coluna_etapa = 'etapa_3' THEN coluna_id_cliente END) AS compradores,
           ROUND(1.0 * COUNT(DISTINCT CASE WHEN coluna_etapa = 'etapa_3' THEN coluna_id_cliente END)
                 / COUNT(DISTINCT coluna_id_cliente), 4) AS taxa_conversao
    FROM tabela_eventos
    GROUP BY coluna_variante
''')

In [ ]:
consulta('''
    WITH receita_mensal AS (
        SELECT strftime('%Y-%m', coluna_data_evento) AS ano_mes,
               SUM(coluna_valor_pedido)              AS receita,
               COUNT(DISTINCT coluna_id_cliente)     AS compradores
        FROM tabela_eventos
        WHERE coluna_etapa = 'etapa_3'
        GROUP BY ano_mes
    )
    SELECT ano_mes,
           ROUND(receita, 2) AS receita,
           compradores,
           ROUND(SUM(receita) OVER (PARTITION BY substr(ano_mes, 1, 4) ORDER BY ano_mes), 2) AS acumulado_ano,
           ROUND(receita / LAG(receita) OVER (ORDER BY ano_mes) - 1, 4)                     AS var_mes_anterior,
           RANK() OVER (ORDER BY receita DESC)                                              AS ranking_receita
    FROM receita_mensal
    ORDER BY ano_mes
''').head(14)

In [ ]:
conn.close()

**Equivalências SQL ↔ pandas**

| SQL | pandas |
|---|---|
| `SELECT col_1, col_2` | `df[['col_1', 'col_2']]` |
| `WHERE cond` | `df.loc[cond]` / `df.query('...')` |
| `ORDER BY col DESC LIMIT 10` | `df.sort_values('col', ascending=False).head(10)` ou `df.nlargest(10, 'col')` |
| `GROUP BY col` + `COUNT`/`SUM`/`AVG` | `df.groupby('col').agg(...)` |
| `HAVING COUNT(*) >= 30` | `.agg(...).query('n >= 30')` |
| `COUNT(DISTINCT col)` | `.nunique()` |
| `LEFT JOIN ... ON` | `df.merge(outra, on='chave', how='left')` |
| `CASE WHEN ... THEN ... END` | `np.where(...)` / `np.select(...)` |
| `SUM(...) OVER (ORDER BY ...)` | `.cumsum()` |
| `LAG(...) OVER (ORDER BY ...)` | `.shift()` |
| `RANK() OVER (ORDER BY ...)` | `.rank(method='min', ascending=False)` |
| `UNION ALL` | `pd.concat([df_1, df_2])` |

⚠️ **O que observar**
- Ordem lógica de execução: `FROM` → `JOIN` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY` → `LIMIT`. Por isso, um apelido criado no `SELECT` não pode ser usado no `WHERE`.
- `NULL` não é igual a nada, nem a outro `NULL`: use `IS NULL` / `IS NOT NULL`. `COUNT(coluna)` ignora `NULL`, enquanto `COUNT(*)` conta todas as linhas.
- Divisão entre inteiros pode truncar o resultado em alguns bancos (`1 / 2 = 0`). Multiplique por `1.0` antes de dividir.
- Um `JOIN` com chave duplicada multiplica as linhas. Confira a contagem antes e depois.
- Funções de data e de texto variam entre bancos (`strftime` no SQLite, `DATE_TRUNC` no PostgreSQL, `FORMAT` no SQL Server).

---
# 3. Primeira inspeção

## 3.1 Estrutura da tabela

**O que faz:** mostra o tamanho da tabela, os nomes das colunas e algumas linhas de exemplo.

**Quando usar:** logo após a leitura, para confirmar que o arquivo foi lido corretamente (colunas separadas, cabeçalho certo).

⚠️ `head()` mostra só o início do arquivo, que pode ter sido ordenado. `sample()` sorteia linhas e dá uma visão mais representativa.

In [ ]:
print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')
df.columns.tolist()

In [ ]:
df.sample(10, random_state=SEED)   # 10 linhas aleatórias (df.head(10) / df.tail(10) mostram início e fim)

## 3.2 Tipos de dados

**O que faz:** `df.info()` mostra, em uma só saída, o tipo de cada coluna, a quantidade de valores não nulos e o uso de memória.

**Quando usar:** sempre. O tipo determina quais operações e gráficos fazem sentido.

| dtype (pandas) | Significado | Atenção |
|---|---|---|
| `int64` / `float64` | números | colunas inteiras com nulos viram `float` |
| `object` / `str` | texto (ou mistura de tipos) | números com vírgula ou símbolos (`R$`) são lidos como texto |
| `bool` | verdadeiro/falso | `True` = 1 e `False` = 0, então `sum()` conta e `mean()` dá a proporção |
| `datetime64` | datas | datas costumam ser lidas como texto e precisam de conversão (Seção 4.2) |
| `category` | categorias (eficiente, permite ordem) | útil para variáveis ordinais |

In [ ]:
df.info()

### Tipos estatísticos de variável

O dtype diz como o dado está **armazenado**. O tipo estatístico diz como ele deve ser **analisado**.

| Tipo | Subtipo | Exemplo genérico | Resumos e gráficos indicados |
|---|---|---|---|
| **Quantitativa** | Discreta: contagens, valores inteiros finitos | nº de itens, nº de ocorrências | média, mediana, contagem por valor, histograma com `discrete=True` |
| | Contínua: medições, infinitos valores possíveis | valor, tempo, peso | média, mediana, desvio padrão, quartis, histograma, boxplot |
| **Qualitativa** (categórica) | Nominal: sem ordem | cor, região, tipo | frequência, moda, gráfico de barras |
| | Ordinal: com ordem | baixo < médio < alto | frequência na ordem correta, mediana (posição) |

⚠️ **O que observar**
- Códigos numéricos (CEP, ID, código de produto) são **categóricos**, mesmo armazenados como número. Não calcule média deles.
- Uma variável contínua pode aparecer arredondada como inteiro e continua sendo contínua.
- Em escalas ordinais, a distância entre categorias não é necessariamente igual.

### Selecionar colunas por tipo

**Quando usar:** para aplicar a mesma análise a todas as colunas numéricas ou a todas as categóricas.

⚠️ `select_dtypes('float')` **não** inclui `int`. Use `'number'` para pegar todas as numéricas. Para texto, `exclude='number'` funciona tanto no pandas 2 (`object`) quanto no pandas 3 (`str`).

In [ ]:
cols_num = df.select_dtypes(include='number').columns.drop('coluna_id', errors='ignore').tolist()
cols_cat = df.select_dtypes(exclude=['number', 'datetime']).columns.tolist()

print('Numéricas :', cols_num)
print('Categóricas/texto:', cols_cat)

## 3.3 Valores nulos

**O que faz:** conta os valores ausentes em cada coluna, em número absoluto e em percentual, ordenando os casos mais graves primeiro.

**Quando usar:** sempre, antes de qualquer cálculo. Nulos afetam médias, contagens e testes.

**Como funciona:** `df.isna()` gera uma tabela de `True`/`False`. Como `True` vale 1, `.sum()` conta os nulos e `.mean()` dá a proporção diretamente.

In [ ]:
nulos = pd.DataFrame({
    'n_nulos': df.isna().sum(),
    'pct_nulos': df.isna().mean() * 100,
}).sort_values('pct_nulos', ascending=False)

nulos[nulos['n_nulos'] > 0]

In [ ]:
# Visualização: útil quando há muitas colunas
pct = df.isna().mean().sort_values(ascending=False)
pct = pct[pct > 0]

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=pct.index, y=pct.values * 100, ax=ax, color='steelblue')
ax.set_ylabel('% de nulos')
ax.set_title('Percentual de dados faltantes por coluna')
plt.xticks(rotation=45, ha='right')
plt.show()

⚠️ **O que observar**
- Colunas com **100% de nulos** não têm informação e podem ser descartadas.
- Colunas com **mais de 50% de nulos** exigem cautela, porque qualquer conclusão vale só para quem preencheu.
- Pergunte **por que** o dado está faltando. Se o nulo não é aleatório (por exemplo, quem ganha mais deixa de informar a renda), remover essas linhas enviesa a análise.
- `describe()`, `mean()` e os gráficos do seaborn **ignoram os nulos automaticamente**. `value_counts()` também ignora, a menos que se use `dropna=False`.
- O tratamento (remover, preencher ou sinalizar) está na Seção 4.3.

## 3.4 Linhas duplicadas ➕

**O que faz:** identifica linhas repetidas, seja a linha inteira ou a mesma chave (ID) aparecendo mais de uma vez.

**Quando usar:** sempre que cada linha deveria representar uma entidade única (cliente, pedido, resposta).

⚠️ Uma duplicata pode ser legítima (dois pedidos idênticos, por exemplo). Confirme pela chave antes de remover.

In [ ]:
print('Linhas totalmente duplicadas:', df.duplicated().sum())
print('IDs repetidos               :', df['coluna_id'].duplicated().sum())

df[df.duplicated(keep=False)].sort_values('coluna_id').head()   # inspeciona os casos

## 3.5 Valores únicos e consistência das categorias

**O que faz:** mostra quantas categorias cada coluna tem e quais são.

**Quando usar:** para encontrar categorias inconsistentes (`'SP'` vs `'sp'` vs `'São Paulo'`), categorias "lixo" (`'Prefer not to say'`, `'-'`) e colunas com categorias demais.

In [ ]:
df[cols_cat].nunique().sort_values()

In [ ]:
df['coluna_cat'].unique()

⚠️ **O que observar**
- Uma coluna "categórica" com milhares de valores únicos costuma ser texto livre ou **multivalorada** (várias respostas separadas por `;`). Ela precisa ser tratada antes da análise (Seção 4.6).
- Colunas com 1 único valor não carregam informação.
- Para segmentar análises, prefira variáveis com **poucas categorias**. Categorias demais fragmentam os dados.
- As tabelas de frequência (contagem e %) estão na Seção 5.6.

## 3.6 Diagnóstico rápido em uma tabela

**O que faz:** reúne, em uma única tabela por coluna, o tipo, os nulos, os valores únicos e um exemplo de valor. Substitui várias chamadas separadas.

**Quando usar:** como primeira visão de um dataset novo, especialmente quando ele tem muitas colunas.

In [ ]:
def visao_geral(data):
    # Resumo por coluna: tipo, nulos, cardinalidade e exemplo de valor.
    return pd.DataFrame({
        'tipo': data.dtypes.astype(str),
        'n_nulos': data.isna().sum(),
        'pct_nulos': (data.isna().mean() * 100).round(1),
        'n_unicos': data.nunique(),
        'exemplo': data.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })

visao_geral(df)

---
# 4. Limpeza e tratamento

> **Regra de ouro:** cada transformação deve ser **intencional e documentada**. Trabalhe sempre sobre `df`, com `df_original` guardado, e confirme o resultado após cada passo (`value_counts()`, `info()`, `shape`).

## 4.1 Nomes de colunas e colunas irrelevantes

**O que faz:** padroniza nomes (minúsculas, sem espaços) e remove colunas que não servem à análise.

**Quando usar:** quando os nomes têm espaços, acentos ou maiúsculas misturadas, porque isso dificulta escrever `df['...']`.

In [ ]:
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('-', '_'))

# Renomear colunas específicas
df = df.rename(columns={'nome_antigo': 'nome_novo'})          # nomes inexistentes são ignorados

# Remover colunas sem utilidade (ex.: 100% nulas, constantes, fora do escopo)
df = df.drop(columns=['coluna_irrelevante'], errors='ignore')

## 4.2 Conversão de tipos

**O que faz:** corrige colunas lidas com o tipo errado. Os casos mais comuns são números como texto, datas como texto e categorias como texto livre.

**Quando usar:** sempre que `df.info()` mostrar um tipo incoerente com o conteúdo.

In [ ]:
# Texto -> número, no formato brasileiro "1.234,56"
df['coluna_valor'] = pd.to_numeric(
    df['coluna_valor_texto']
      .str.replace('.', '', regex=False)    # 1) remove o separador de milhar
      .str.replace(',', '.', regex=False),  # 2) troca a vírgula decimal por ponto
    errors='coerce',                        # valores inválidos viram NaN em vez de gerar erro
)

# Texto -> data (informe o formato para evitar ambiguidade dia/mês)
df['coluna_data'] = pd.to_datetime(df['coluna_data'], format='%d/%m/%Y')
df['coluna_data_fim'] = pd.to_datetime(df['coluna_data_fim'], format='%d/%m/%Y')

df[['coluna_valor_texto', 'coluna_valor', 'coluna_data', 'coluna_data_fim']].dtypes

⚠️ **O que observar**
- Só `.str.replace(',', '.')` funciona com `"1234,56"`, mas **quebra** com `"1.234,56"`. Remova o separador de milhar primeiro, ou resolva na leitura com `decimal=','` e `thousands='.'`.
- Com `errors='coerce'`, compare a quantidade de nulos antes e depois. Novos nulos indicam valores que não puderam ser convertidos e precisam ser investigados.
- Outros formatos de data: `'%Y-%m-%d'` (ISO), `'%d/%m/%Y %H:%M'`. Quando o formato varia, use `dayfirst=True`.
- `df['coluna'].astype('string')` força o tipo texto (útil para garantir que todos os valores sejam strings). `astype('category')` economiza memória em colunas repetitivas.

## 4.3 Tratamento de valores nulos

**O que faz:** decide o destino dos nulos encontrados na Seção 3.3.

**Quando usar:** depois de entender **por que** os valores faltam. Escolha a estratégia conforme o caso:

| Estratégia | Código | Quando usar | Risco |
|---|---|---|---|
| Remover linhas | `df.dropna(subset=['coluna'])` | poucos nulos (< ~5%) e aleatórios | perda de dados e viés, se o nulo não for aleatório |
| Remover coluna | `df.drop(columns=...)` | coluna quase toda vazia | perder variável útil |
| Preencher categoria | `fillna('Não informado')` | categóricas: o "não informado" vira uma categoria visível | nenhum, se for analisado como grupo próprio |
| Preencher com mediana/moda | `fillna(df['col'].median())` | numéricas, quando um modelo exige dados completos | reduz a variabilidade e cria picos artificiais |
| Sinalizar | `df['col'].isna().astype(int)` | quando o fato de faltar pode ser informativo | nenhum, porque preserva a informação |
| Ignorar | (não fazer nada) | pandas e seaborn já ignoram NaN em cálculos e gráficos | lembrar de reportar o `n` efetivo |

In [ ]:
# Categórica: nulo vira categoria explícita
df['coluna_cat'] = df['coluna_cat'].fillna('Não informado')

# Numérica: sinaliza a ausência e mantém o NaN (os cálculos ignoram NaN)
df['coluna_num_ausente'] = df['coluna_num'].isna().astype(int)

# Remover linhas sem valor em colunas essenciais (exemplo; descomente se fizer sentido)
# df = df.dropna(subset=['coluna_num'])

df['coluna_cat'].value_counts(dropna=False)   # confirma o resultado

## 4.4 Remoção de duplicados ➕

In [ ]:
n_antes = len(df)
df = df.drop_duplicates()                                  # linha inteira igual
# df = df.drop_duplicates(subset=['coluna_id'], keep='first')  # mesma chave
print(f'Removidas {n_antes - len(df)} linhas duplicadas')

## 4.5 Recodificação de categorias

**O que faz:** renomeia, agrupa e **ordena** categorias.

**Quando usar:**
- rótulos longos, inconsistentes ou pouco informativos (`'Prefer not to say'`, `'-'`);
- categorias ordinais que precisam aparecer na ordem lógica, e não alfabética, em tabelas e gráficos;
- muitas categorias raras que poluem a análise.

In [ ]:
# 1) Mapear valores (map com dicionário: valores ausentes no dicionário viram NaN)
mapa = {'baixo': 'Baixo', 'médio': 'Médio', 'alto': 'Alto'}
df['coluna_ordinal'] = df['coluna_ordinal'].map(mapa)

# 2) Definir a ORDEM das categorias ordinais
ordem = ['Baixo', 'Médio', 'Alto']
df['coluna_ordinal'] = pd.Categorical(df['coluna_ordinal'], categories=ordem, ordered=True)

df['coluna_ordinal'].value_counts().sort_index()   # sort_index respeita a ordem definida

In [ ]:
# 3) Manter apenas categorias relevantes (ex.: com mais de 1% dos registros)
freq = df['coluna_cat'].value_counts(normalize=True)
categorias_relevantes = freq[freq > 0.01].index
df_relevantes = df[df['coluna_cat'].isin(categorias_relevantes)]

# 4) ... ou agrupar as categorias raras em 'Outros'
df['coluna_cat_agrupada'] = df['coluna_cat'].where(df['coluna_cat'].isin(categorias_relevantes), 'Outros')

⚠️ **O que observar**
- `map` transforma em `NaN` tudo o que não estiver no dicionário. Para trocar apenas alguns valores e manter o resto, use `replace`. Confira sempre com `value_counts(dropna=False)`.
- Antes de descartar categorias pouco informativas (`'Não informado'`, `'Prefer not to say'`), meça quanto elas representam. Se forem poucas (< ~5%), ignorá-las raramente muda as conclusões.
- Uma alternativa improvisada para ordenar categorias é criar rótulos que já se ordenam alfabeticamente (`'[18 - 24]'`, `'[25 - 34]'`). `pd.Categorical(ordered=True)` é mais robusto.

## 4.6 Colunas de texto e colunas multivaloradas

**O que faz:** limpa texto e extrai informação de colunas com vários valores na mesma célula (ex.: `"item_1;item_3"`).

**Quando usar:** quando `nunique()` é muito alto porque cada linha tem uma **combinação** de respostas.

In [ ]:
# Limpeza básica de texto
df['coluna_cat_2'] = df['coluna_cat_2'].str.strip().str.upper()

# Flag: a linha contém um valor específico? (na=False trata nulos como "não contém")
df['contem_item_1'] = df['coluna_multivalor'].str.contains('item_1', na=False).astype(int)

# Contagem de cada valor individual: split + explode (uma linha por valor)
contagem_itens = (df['coluna_multivalor']
                  .str.split(';')
                  .explode()
                  .value_counts())
contagem_itens

⚠️ `str.contains('item_1')` também encontra `'item_10'`. Para correspondência exata, use `explode` ou uma regex com limites, como `r'(^|;)item_1(;|$)'`. Outros métodos úteis: `str.startswith`, `str.endswith`, `str.lower`, `str.len`, `str.extract`.

## 4.7 Filtros

**O que faz:** seleciona linhas por condição.

**Quando usar:** para recortar o escopo (período, segmento), excluir valores inválidos ou isolar casos para inspeção.

In [ ]:
filtro_valor = df['coluna_num_2'] > 100
filtro_cat = df['coluna_cat'].isin(['A', 'B'])
filtro_texto = df['coluna_cat'].str.startswith('A')

df_filtrado = df.loc[filtro_valor & filtro_cat].copy()   # & = E, | = OU, ~ = NÃO (use parênteses)

# Equivalentes úteis
df.loc[df['coluna_num'].between(10, 20)]                 # intervalo inclusivo
df.query('coluna_num_2 > 100 and coluna_cat == "A"')      # sintaxe de texto

print(f'{len(df_filtrado)} de {len(df)} linhas ({len(df_filtrado) / len(df):.1%})')

⚠️ **O que observar**
- Dar nomes às condições (`filtro_valor`, `filtro_cat`) deixa a lógica legível e permite combiná-las.
- Use `.copy()` ao criar um DataFrame filtrado que será modificado depois. Isso evita o `SettingWithCopyWarning` e efeitos colaterais.
- Sempre reporte **quantas linhas** o filtro removeu.

## 4.8 Criação de colunas

**O que faz:** deriva novas variáveis (totais, indicadores, faixas) das existentes.

**Quando usar:** quando a pergunta de negócio depende de uma medida que não existe pronta.

In [ ]:
# Soma de colunas
df['coluna_total'] = df['coluna_valor'] + df['coluna_num_2']                 # NaN em qualquer parcela -> NaN
# df['coluna_total'] = df[['coluna_valor', 'coluna_num_2']].sum(axis=1)     # trata NaN como 0

# Indicador binário a partir de uma condição
df['flag_cat_a'] = (df['coluna_cat'] == 'A').astype(int)

# Condição com dois resultados
df['nivel'] = np.where(df['coluna_num_2'] > df['coluna_num_2'].median(), 'acima_mediana', 'abaixo_mediana')

# Faixas (discretização) de uma variável numérica
df['faixa_num'] = pd.cut(df['coluna_num'], bins=[0, 5, 10, 20, np.inf],
                         labels=['0-5', '5-10', '10-20', '20+'])
# pd.qcut(df['coluna_num'], q=4) -> faixas com a mesma quantidade de registros (quartis)

df[['coluna_valor', 'coluna_num_2', 'coluna_total', 'flag_cat_a', 'nivel', 'faixa_num']].head()

## 4.9 Transformação logarítmica

**O que faz:** comprime valores altos e reduz a assimetria à direita.

**Quando usar:**
- variáveis com **cauda longa à direita** (renda, preços, tempos, contagens grandes);
- quando um teste ou modelo pressupõe normalidade;
- quando poucos valores muito altos dominam o gráfico.

In [ ]:
# log(0) é indefinido: filtre os valores > 0 ou use log1p = log(1 + x)
df['log_coluna_num'] = np.log(df['coluna_num'].where(df['coluna_num'] > 0))
# df['log_coluna_num'] = np.log1p(df['coluna_num'])   # alternativa quando há zeros

df[['coluna_num', 'log_coluna_num']].skew()   # assimetria antes x depois (ver Seção 5.4)

⚠️ **O que observar**
- Resultados na escala log são interpretados em termos **relativos**: uma diferença de 0,1 no log equivale a cerca de 10% de diferença no valor original.
- Para apresentar ao público, volte à escala original com `np.exp()`, ou use eixo logarítmico no gráfico: `ax.set_xscale('log')`.

## 4.10 Junção de tabelas ➕

**O que faz:** combina duas tabelas por uma chave em comum (`merge`), por exemplo para trazer atributos de uma tabela de dimensão para a tabela principal.

**Quando usar:** quando a informação necessária está espalhada em mais de uma tabela ou arquivo.

| Tipo (`how=`) | Mantém | Uso típico |
|---|---|---|
| `'left'` | todas as linhas da esquerda; sem correspondência → NaN | enriquecer a tabela principal sem perder registros |
| `'inner'` | só as chaves presentes nas duas | analisar apenas o que existe nos dois lados |
| `'outer'` | todas as chaves dos dois lados | auditar diferenças entre tabelas |
| `'right'` | todas as linhas da direita | o mesmo que `left`, invertendo as tabelas |

In [ ]:
n_antes = len(df)
df_com_categoria = df.merge(tabela_categorias, on='coluna_cat', how='left',
                            validate='many_to_one')   # erro se a chave se repetir na tabela de dimensão

print(f'Linhas antes: {n_antes} | depois: {len(df_com_categoria)}')
print('Linhas sem correspondência:', df_com_categoria['coluna_grupo'].isna().sum())
df_com_categoria[['coluna_id', 'coluna_cat', 'coluna_grupo', 'coluna_meta']].head()

In [ ]:
# Diagnóstico: quais chaves existem só de um lado?
diagnostico = df[['coluna_cat']].drop_duplicates().merge(
    tabela_categorias[['coluna_cat']], on='coluna_cat', how='outer', indicator=True)
diagnostico

In [ ]:
# Consultar todos os registros de uma chave específica em outra tabela
valor_id = eventos['coluna_id_cliente'].iloc[0]
eventos[eventos['coluna_id_cliente'] == valor_id]

⚠️ **O que observar**
- Confira **sempre** o número de linhas antes e depois. Um aumento em um `left` join significa chave duplicada na tabela da direita. `validate='many_to_one'` (ou `'one_to_one'`) transforma esse risco em erro explícito.
- `indicator=True` cria a coluna `_merge` (`left_only`, `right_only`, `both`), ótima para auditar chaves sem correspondência.
- As chaves precisam ter o **mesmo tipo e formato** nas duas tabelas (`'001'` ≠ `1`, `'A '` ≠ `'A'`). Padronize antes de juntar (Seções 4.2 e 4.6).
- Colunas com o mesmo nome nos dois lados recebem os sufixos `_x` e `_y`. Use `suffixes=('_principal', '_dimensao')` para nomes claros.
- Para empilhar tabelas com as mesmas colunas (por exemplo, um arquivo por mês), use `pd.concat([df_1, df_2], ignore_index=True)` (Seção 2.1).

---
# 5. Estatística descritiva

**Estatística descritiva** resume e descreve os dados observados. **Estatística inferencial** (Seção 12) generaliza da amostra para a população.

| Conceito | Definição | Notação |
|---|---|---|
| População | conjunto completo de interesse | parâmetros: $\mu$ (média), $\sigma$ (desvio) |
| Amostra | fração representativa e não tendenciosa da população | estatísticas: $\bar{x}$ (média), $s$ (desvio) |
| Intervalo de confiança | faixa plausível para um parâmetro populacional | Seção 12.9 |

## 5.1 Visão geral: `describe()`

**O que faz:** mostra contagem, média, desvio padrão, mínimo, quartis e máximo de todas as colunas numéricas de uma vez.

**Quando usar:** como primeiro resumo numérico. Selecione só as colunas que fazem sentido (IDs e códigos não!).

In [ ]:
df[['coluna_num', 'coluna_num_2', 'coluna_valor']].describe().round(2)

In [ ]:
df.describe(exclude='number')   # para texto/categorias: count, unique, top (moda) e freq

⚠️ **O que observar**
- `count` menor que o total de linhas indica nulos. `describe()` **ignora NaN**.
- **Mediana (50%) menor que a média** indica que valores altos estão puxando a média para cima, ou seja, há **assimetria à direita**. É comum em renda, preços e tempos.
- Mínimos ou máximos impossíveis (idade negativa, % acima de 100) indicam erro de dado.
- Colunas cuja média não faz sentido (IDs, códigos) não devem entrar no resumo.

## 5.2 Medidas de tendência central

| Medida | Código | Quando usar | Cuidado |
|---|---|---|---|
| Média $\bar{x} = \frac{\sum x_i}{n}$ | `.mean()` | distribuições simétricas | é sensível a outliers |
| Mediana (valor do meio) | `.median()` | dados assimétricos ou com outliers | ignora a magnitude dos extremos |
| Moda (valor mais frequente) | `.mode()` | categóricas, ou numéricas discretas | pode haver mais de uma; retorna uma Series |

In [ ]:
col = df['coluna_num']

print(f'Média  : {col.mean():.2f}')
print(f'Mediana: {col.median():.2f}')
print(f'Moda   : {col.mode().tolist()}')            # lista: pode haver empate
print(f"Moda (categórica): {df['coluna_cat'].mode().iloc[0]}")

## 5.3 Medidas de dispersão

| Medida | Fórmula / código | Interpretação |
|---|---|---|
| Amplitude | `max() - min()` | distância entre os extremos (muito sensível a outliers) |
| Variância | `.var()` | dispersão em torno da média, **na unidade ao quadrado** |
| Desvio padrão | `.std()` = $\sqrt{\text{variância}}$ | dispersão típica em torno da média, **na unidade original** |
| Coeficiente de variação | $CV = \frac{s}{\bar{x}}$ | dispersão relativa à média; permite comparar variáveis com escalas diferentes |
| Erro padrão da média | $EP = \frac{s}{\sqrt{n}}$ | precisão da média amostral como estimativa da média populacional |
| IQR | $Q3 - Q1$ | dispersão dos 50% centrais (robusta a outliers) |

**Amostral ou populacional?** Use o amostral ($n-1$ no denominador) quando os dados forem uma amostra, que é o caso mais comum. Use o populacional ($N$) quando tiver a população inteira.

In [ ]:
col = df['coluna_num'].dropna()

amplitude = col.max() - col.min()
var_amostral, var_pop = col.var(), col.var(ddof=0)   # pandas: ddof=1 (amostral) é o padrão
dp_amostral, dp_pop = col.std(), col.std(ddof=0)
cv = col.std() / col.mean()
erro_padrao = stats.sem(col)                          # = col.std() / np.sqrt(len(col))

print(f'Amplitude              : {amplitude:.2f}')
print(f'Variância (amostral)   : {var_amostral:.2f} | populacional: {var_pop:.2f}')
print(f'Desvio padrão (amostral): {dp_amostral:.2f} | populacional: {dp_pop:.2f}')
print(f'Coef. de variação      : {cv:.1%}')
print(f'Erro padrão da média   : {erro_padrao:.3f}')

# Equivalente com a biblioteca padrão: st.variance / st.pvariance / st.stdev / st.pstdev

⚠️ **O que observar**
- `numpy.std()` usa **ddof=0** (populacional) por padrão, enquanto `pandas.std()` usa **ddof=1** (amostral). Isso é uma fonte comum de diferenças entre resultados.
- **CV**: abaixo de ~15% indica baixa dispersão; de 15% a 30%, moderada; acima de 30%, alta. São referências, não regras. O CV só faz sentido em variáveis com zero absoluto e valores positivos.
- O erro padrão diminui com $\sqrt{n}$. Amostras maiores produzem médias mais precisas.

## 5.4 Medidas separatrizes e forma da distribuição

**Quartis** dividem os dados em 4 partes com 25% cada (Q1 = 25%, Q2 = mediana, Q3 = 75%). **Percentis** dividem em 100 partes.

**Assimetria (skewness)**: o lado para onde vai a "cauda"
- `= 0`: simétrica. Média ≈ mediana ≈ moda.
- `> 0`: cauda à direita (positiva). Média > mediana.
- `< 0`: cauda à esquerda (negativa). Média < mediana.
- Referência prática: |assimetria| < 0,5 indica distribuição aproximadamente simétrica; entre 0,5 e 1, moderada; > 1, forte.

**Curtose**: o quanto a distribuição é "pontuda" e tem caudas pesadas, comparada à normal
- `= 0`: **mesocúrtica** (como a normal).
- `> 0`: **leptocúrtica**, com pico alto e caudas pesadas (mais valores extremos).
- `< 0`: **platicúrtica**, achatada.

In [ ]:
col = df['coluna_num'].dropna()

q1, q2, q3 = col.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1
p90, p99 = col.quantile([0.90, 0.99])

print(f'Q1={q1:.2f} | Mediana={q2:.2f} | Q3={q3:.2f} | IQR={iqr:.2f}')
print(f'P90={p90:.2f} | P99={p99:.2f}')
print(f'Assimetria: {col.skew():.2f}')
print(f'Curtose   : {col.kurt():.2f}')

⚠️ `pandas.kurt()` e `scipy.stats.kurtosis(fisher=True)` retornam a curtose **em excesso** (normal = 0). Com `fisher=False`, a normal vale 3. O pandas aplica correção de viés amostral e o scipy, por padrão, não. Por isso, os valores podem diferir levemente.

## 5.5 Tabela descritiva completa (função reutilizável)

**O que faz:** reúne em **uma tabela** todas as medidas acima (n, média, desvio, CV, quartis, assimetria e curtose) para todas as colunas numéricas.

**Quando usar:** é a tabela-padrão de qualquer relatório. Aplicada por grupo, ela faz a **descritiva segmentada**.

In [ ]:
def descritivas(data):
    # Estatísticas descritivas completas das colunas numéricas.
    variaveis = data.select_dtypes(include='number')
    desc = variaveis.describe().T
    desc['CV'] = desc['std'] / desc['mean']
    desc['IQR'] = desc['75%'] - desc['25%']
    desc['assimetria'] = variaveis.skew()
    desc['curtose'] = variaveis.kurt()
    colunas = ['count', 'mean', 'std', 'CV', 'min', '25%', '50%', '75%', 'max', 'IQR', 'assimetria', 'curtose']
    return desc[colunas].round(2)

cols_analise = ['coluna_num', 'coluna_num_2', 'coluna_valor']
descritivas(df[cols_analise])

In [ ]:
# Descritiva segmentada: a mesma tabela para cada categoria
df.groupby('coluna_cat')[cols_analise].apply(descritivas)

In [ ]:
# Versão compacta: apenas as métricas escolhidas, por grupo
df.groupby('coluna_cat')['coluna_num'].agg(['count', 'mean', 'median', 'std']).round(2)

## 5.6 Frequências de variáveis categóricas

**O que faz:** conta quantas vezes cada categoria aparece, em número e em percentual.

**Quando usar:** é o equivalente do `describe()` para variáveis categóricas.

In [ ]:
def tabela_frequencia(serie, dropna=False):
    # Contagem e percentual de cada categoria, incluindo nulos por padrão.
    return pd.DataFrame({
        'n': serie.value_counts(dropna=dropna),
        'pct': serie.value_counts(normalize=True, dropna=dropna).mul(100).round(2),
    })

tabela_frequencia(df['coluna_cat'])

In [ ]:
# Formatação percentual apenas para exibição
df['coluna_cat'].value_counts(normalize=True).map('{:.2%}'.format)

In [ ]:
# Combinação de duas categóricas (contagem de cada par); para tabela cruzada, ver Seção 8.4
df[['coluna_ordinal', 'coluna_cat_2']].value_counts().sort_index()

⚠️ **O que observar**
- `value_counts()` **exclui nulos** por padrão. Use `dropna=False` para saber quantos não responderam.
- `value_counts()` ordena pela frequência. Para categorias ordinais, use `.sort_index()`.
- Quanto mais colunas combinadas, mais fragmentados ficam os dados e mais difícil é interpretá-los.

## 5.7 Padronização: escore Z

**O que faz:** expressa cada valor em **quantos desvios padrão** ele está da média: $Z = \frac{x - \mu}{\sigma}$. Após a padronização, a variável tem média 0 e desvio 1.

**Quando usar:**
- para comparar valores de variáveis em escalas diferentes;
- para identificar valores extremos (|Z| > 3, Seção 7);
- para calcular probabilidades sob a curva normal.

In [ ]:
df['z_coluna_num'] = stats.zscore(df['coluna_num'], nan_policy='omit')
# equivalente: (df['coluna_num'] - df['coluna_num'].mean()) / df['coluna_num'].std(ddof=0)

# Probabilidade acumulada até um valor Z, sob a normal padrão
z = 1.5
print(f'P(Z <= {z}) = {stats.norm.cdf(z):.4f}')     # proporção esperada de valores abaixo de z
print(f'P(Z >  {z}) = {stats.norm.sf(z):.4f}')

⚠️ Probabilidades calculadas pela curva normal só são válidas se a variável for **aproximadamente normal**. Verifique isso antes (Seções 6.4 e 12.3).

---
# 6. Análise univariada (gráficos)

Os números da Seção 5 resumem a distribuição. Os gráficos mostram sua **forma**: picos, caudas, lacunas, valores arredondados e grupos misturados.

| Gráfico | Mostra | Melhor para |
|---|---|---|
| Histograma | frequência por intervalo (*bins*) | forma geral da distribuição |
| KDE | curva suavizada da densidade | forma e comparação de grupos |
| Boxplot | quartis, mediana e outliers | resumo compacto e comparação entre muitos grupos |
| Violino | KDE + boxplot | forma e quartis ao mesmo tempo |
| Barras / countplot | frequência de categorias | variáveis categóricas |

## 6.1 Histograma

**O que faz:** agrupa os valores em intervalos (*bins*) e mostra quantos caem em cada um.

**Quando usar:** é o primeiro gráfico para qualquer variável numérica.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df, x='coluna_num', binwidth=1, kde=True, ax=ax)   # binwidth: largura de cada barra
ax.set_title('Distribuição de coluna_num')
plt.show()

**Parâmetros-chave de `sns.histplot`**

| Parâmetro | Uso |
|---|---|
| `bins=30` | número de barras |
| `binwidth=1` | largura de cada barra; para inteiros (anos, idades), `1` faz cada barra = um valor |
| `discrete=True` | centraliza as barras em valores inteiros |
| `kde=True` | sobrepõe a curva de densidade |
| `stat='density'` / `'percent'` / `'probability'` | eixo Y relativo, em vez de contagem |
| `hue='coluna_cat'` | um histograma por grupo (Seção 8.3) |
| `log_scale=True` | eixo X em escala logarítmica |

⚠️ **O que observar**
- O número de *bins* muda a aparência do gráfico. Teste mais de um valor.
- **Picos regulares** em valores "redondos" (múltiplos de 5 ou 10) indicam **arredondamento** em dados declarados. As pessoas não lembram o valor exato e arredondam. Confirme com:

In [ ]:
df['coluna_num'].round().value_counts().head(10)   # valores redondos no topo indicam arredondamento

## 6.2 KDE e rugplot

**O que faz:** o **KDE** (*Kernel Density Estimate*) estima uma curva suave da distribuição sem supor nenhuma forma específica. O **rugplot** marca cada observação como um traço no eixo e mostra onde os dados estão. O KDE é construído somando uma pequena curva normal centrada em cada um desses pontos.

**Quando usar:** o KDE serve para comparar formas de distribuição entre grupos. O rugplot serve para ver os dados individuais quando há poucos pontos.

⚠️ O KDE pode "inventar" densidade em regiões impossíveis (valores negativos, por exemplo). `bw_adjust` controla a suavidade: valores menores que 1 dão mais detalhe e mais ruído, e valores maiores que 1 dão uma curva mais suave.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.kdeplot(data=df, x='coluna_num', fill=True, bw_adjust=1, ax=ax)
sns.rugplot(data=df, x='coluna_num', ax=ax, alpha=0.3)
plt.show()

## 6.3 Boxplot, violino e painel completo

**Boxplot**: a caixa vai de Q1 a Q3 e a linha interna é a mediana. Os "bigodes" chegam até o último ponto dentro de `whis × IQR` (padrão 1,5). Os pontos além disso são marcados como outliers (cercas de Tukey, Seção 7.1).

**Violino**: mostra o KDE espelhado com um boxplot interno. Reúne **forma + quartis** em um só gráfico.

**Quando usar:** o boxplot serve para resumir e comparar muitos grupos. O violino serve quando a forma importa, por exemplo em distribuições com dois picos, que o boxplot esconde.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(14, 4))

# Boxplot com os pontos individuais por cima (jitter espalha os pontos para não se sobreporem)
sns.boxplot(data=df, x='coluna_num', whis=1.5, ax=axes[0], showfliers=False)
sns.stripplot(data=df, x='coluna_num', ax=axes[0], color='black', alpha=0.25, size=3, jitter=0.3)
axes[0].set_title('Boxplot + pontos')

sns.violinplot(data=df, x='coluna_num', ax=axes[1], inner='quartile')
axes[1].set_title('Violino (linhas = quartis)')
plt.show()

In [ ]:
def painel_distribuicao(data, coluna, binwidth=None):
    # Histograma + KDE com quartis e média, e boxplot alinhado no mesmo eixo X.
    serie = data[coluna].dropna()
    fig, (ax_hist, ax_box) = plt.subplots(nrows=2, sharex=True, figsize=(12, 6),
                                          gridspec_kw={'height_ratios': [3, 1]})
    sns.histplot(serie, binwidth=binwidth, kde=True, ax=ax_hist)
    for q in serie.quantile([0.25, 0.5, 0.75]):
        ax_hist.axvline(q, color='#453698', linestyle='--', linewidth=2)
    ax_hist.axvline(serie.mean(), color='red', linewidth=1.5, label=f'média = {serie.mean():.2f}')
    ax_hist.legend()
    sns.boxplot(x=serie, ax=ax_box)
    fig.suptitle(f'{coluna} | mediana = {serie.median():.2f} | assimetria = {serie.skew():.2f}')
    plt.show()

painel_distribuicao(df, 'coluna_num', binwidth=1)

⚠️ **O que observar**
- Caixa "espremida" de um lado e muitos outliers do outro indicam distribuição assimétrica. Nesse caso, as cercas de Tukey marcam outliers demais (Seção 7).
- Pontos individuais sobre o boxplot mostram **onde** os dados se concentram, o que a caixa sozinha esconde.
- `swarmplot` é uma alternativa ao `stripplot` que não sobrepõe os pontos, mas fica lento com milhares de observações.

## 6.4 Normalidade visual

**O que faz:** compara a distribuição observada com uma curva normal de mesma média e desvio padrão.

**Quando usar:** antes de usar métodos que pressupõem normalidade (testes paramétricos, probabilidades pelo escore Z).

**Distribuição normal (gaussiana)**: simétrica, unimodal, com média = mediana = moda e caudas que nunca tocam o eixo. Ela é definida por dois parâmetros: a **média** (posição do pico) e o **desvio padrão** (largura). Regra empírica:
- ~68% dos dados ficam a até 1 desvio padrão da média;
- ~95% ficam a até 2 desvios;
- ~99,7% ficam a até 3 desvios.

In [ ]:
def comparar_com_normal(data, coluna):
    # Histograma (densidade) com a curva normal teórica sobreposta e QQ-plot.
    serie = data[coluna].dropna()
    xs = np.linspace(serie.min(), serie.max(), 500)
    ys = stats.norm.pdf(xs, loc=serie.mean(), scale=serie.std())

    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(14, 4))
    sns.histplot(serie, stat='density', kde=True, ax=ax1)
    ax1.plot(xs, ys, color='green', linewidth=2, label='normal teórica')
    ax1.legend()
    stats.probplot(serie, dist='norm', plot=ax2)   # QQ-plot: pontos sobre a reta indicam normalidade
    ax2.set_title('QQ-plot')
    fig.suptitle(f'{coluna} | assimetria = {serie.skew():.2f} | curtose = {serie.kurt():.2f}')
    plt.show()

comparar_com_normal(df, 'coluna_num')
comparar_com_normal(df, 'log_coluna_num')   # após a transformação log (Seção 4.9)

⚠️ **O que observar**
- **QQ-plot** ➕: pontos sobre a reta indicam normalidade. Uma curva em "U" ou em "S" indica assimetria ou caudas pesadas.
- Referência: uma normal tem assimetria ≈ 0 e curtose ≈ 0. Quanto mais longe disso, menos normal é a variável.
- A transformação log costuma aproximar da normalidade distribuições com cauda à direita, mas raramente as torna perfeitamente normais.
- O teste formal de normalidade (Shapiro-Wilk) está na Seção 12.3.

## 6.5 Variáveis categóricas

**O que faz:** mostra a frequência de cada categoria.

**Quando usar:** para toda variável categórica. Barras **horizontais ordenadas** são as mais legíveis quando os rótulos são longos ou há muitas categorias.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(14, 4))

# countplot: conta direto do DataFrame (order define a ordem das barras)
sns.countplot(data=df, x='coluna_ordinal', order=['Baixo', 'Médio', 'Alto'], ax=axes[0])
axes[0].set_title('Categoria ordinal (ordem lógica)')

# barras horizontais ordenadas por frequência (top N)
top = df['coluna_cat'].value_counts().head(10)
sns.barplot(x=top.values, y=top.index, ax=axes[1], color='steelblue')
axes[1].bar_label(axes[1].containers[0], fmt='%d')   # rótulo com o valor em cada barra
axes[1].set_title('Frequência por categoria (top 10)')
plt.tight_layout()
plt.show()

⚠️ Prefira barras a gráficos de pizza: comparar comprimentos é mais fácil do que comparar ângulos. Em variáveis ordinais, mantenha a ordem lógica e não ordene por frequência.

---
# 7. Outliers

**Outlier** é uma observação muito distante das demais. Ele pode ser um **erro** (digitação, unidade errada), um **caso legítimo raro** (um cliente muito grande) ou um sinal de **outra população** misturada nos dados.

> Detectar é técnico. **Decidir o que fazer** depende do conhecimento do negócio.

## 7.1 Método IQR (cercas de Tukey)

**O que faz:** marca como outlier todo valor abaixo de $Q1 - k \cdot IQR$ ou acima de $Q3 + k \cdot IQR$. É o mesmo critério do boxplot, com $k$ = parâmetro `whis`.

**Quando usar:** é o método padrão por ser robusto, já que não depende da média. Use $k = 1{,}5$ como padrão e $k = 3$ para pegar só os casos extremos.

In [ ]:
def limites_iqr(serie, k=1.5):
    # Retorna os limites inferior e superior das cercas de Tukey.
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

lim_inf, lim_sup = limites_iqr(df['coluna_num_2'])
outliers_iqr = df[(df['coluna_num_2'] < lim_inf) | (df['coluna_num_2'] > lim_sup)]

print(f'Limites: [{lim_inf:.2f}, {lim_sup:.2f}]')
print(f'Outliers (IQR): {len(outliers_iqr)} ({len(outliers_iqr) / len(df):.1%})')

In [ ]:
# Visualmente: o parâmetro whis do boxplot é o mesmo k
fig, axes = plt.subplots(ncols=2, figsize=(14, 3), sharex=True)
sns.boxplot(data=df, x='coluna_num_2', whis=1.5, ax=axes[0]).set_title('whis = 1,5 (padrão)')
sns.boxplot(data=df, x='coluna_num_2', whis=3.0, ax=axes[1]).set_title('whis = 3,0 (só extremos)')
plt.show()

## 7.2 Método Z-score

**O que faz:** marca como outlier todo valor a mais de 3 desvios padrão da média ($|Z| > 3$).

**Quando usar:** apenas em distribuições **aproximadamente normais**.

⚠️ Os próprios outliers inflam a média e o desvio padrão, o que pode "esconder" outros outliers. Em dados assimétricos, prefira o IQR, ou aplique o Z-score sobre o log da variável.

In [ ]:
z = stats.zscore(df['coluna_num_2'], nan_policy='omit')
outliers_z = df[np.abs(z) > 3]
print(f'Outliers (|Z| > 3): {len(outliers_z)}')

## 7.3 Inspeção dos casos extremos

**O que faz:** lista os maiores e menores valores e as linhas completas desses registros.

**Quando usar:** sempre, antes de decidir o que fazer. Olhe as outras colunas desses registros para entender se são erro ou caso legítimo.

In [ ]:
df.nlargest(10, 'coluna_num_2')        # 10 maiores (nsmallest para os menores)

In [ ]:
limite_negocio = 500                   # limite definido pelo conhecimento do negócio
(df['coluna_num_2'] > limite_negocio).value_counts()   # quantos registros passam do limite

## 7.4 Tratamento

| Estratégia | Código | Quando usar |
|---|---|---|
| **Manter** | - | outliers legítimos e relevantes; use medidas robustas (mediana, IQR, Spearman, testes não paramétricos) |
| **Remover** | `df[df['col'] <= limite]` | erro comprovado ou fora do escopo da análise; **reporte quantos foram removidos** |
| **Limitar (winsorizar)** ➕ | `df['col'].clip(lower, upper)` | manter o registro, mas reduzir sua influência |
| **Transformar** | `np.log(...)` (Seção 4.9) | cauda longa natural (renda, preço) |
| **Analisar à parte** | `df[df['col'] > limite]` | quando os extremos formam outro grupo (Seção 8.5) |

In [ ]:
df_sem_outliers = df[df['coluna_num_2'].between(lim_inf, lim_sup)].copy()
df['coluna_num_2_limitada'] = df['coluna_num_2'].clip(lower=lim_inf, upper=lim_sup)

# Medir o impacto da decisão
impacto = pd.DataFrame({
    'com outliers': df['coluna_num_2'].agg(['count', 'mean', 'median', 'std']),
    'sem outliers': df_sem_outliers['coluna_num_2'].agg(['count', 'mean', 'median', 'std']),
})
impacto

⚠️ **O que observar**
- A **média** e o **desvio padrão** mudam muito com outliers. A **mediana** praticamente não muda. Uma grande diferença entre as duas colunas acima indica o peso dos extremos.
- Outliers distorcem correlação e regressão (Seção 13.2). Sempre compare os resultados com e sem eles.
- As cercas de Tukey funcionam melhor em dados **simétricos**. Em dados assimétricos, elas marcam "outliers" que são apenas a cauda natural. Escolha `k` pelo conhecimento dos dados.
- Nunca remova outliers só para "melhorar" um resultado.

---
# 8. Análise bivariada e multivariada

Escolha a técnica pelo **tipo das duas variáveis**:

| Variáveis | Resumo numérico | Gráfico | Teste (Seção 12) |
|---|---|---|---|
| Numérica × Numérica | correlação (Pearson/Spearman) | dispersão, histograma 2D, jointplot | `pearsonr` / `spearmanr`, regressão |
| Numérica × Categórica | `groupby().agg()` | boxplot, violino, histogramas/KDE por grupo, barras com erro | t, ANOVA, Mann-Whitney, Kruskal |
| Categórica × Categórica | tabela cruzada (`crosstab`) | heatmap da tabela, barras empilhadas | qui-quadrado |
| 3+ variáveis | `pivot_table`, `groupby` múltiplo | `hue`, facetas (`col`/`row`), pairplot, heatmap | - |

## 8.1 Numérica × Numérica: dispersão e correlação

**Correlação** mede o quanto duas variáveis variam **juntas**. Ela vai de **-1** (relação negativa perfeita) a **+1** (relação positiva perfeita), e **0** indica ausência de relação.

| Coeficiente | Mede | Quando usar |
|---|---|---|
| **Pearson** (`r`) | relação **linear** (proporcional) | variáveis contínuas, relação aproximadamente linear, sem outliers fortes |
| **Spearman** (`ρ`) | relação **monotônica** (por ranking: "quem é maior em X também é maior em Y?") | relação não linear mas sempre crescente/decrescente, dados ordinais, presença de outliers |

Referência prática para |r|: < 0,3 fraca; 0,3 a 0,7 moderada; > 0,7 forte.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(14, 5))

sns.scatterplot(data=df, x='coluna_num', y='coluna_num_2', alpha=0.4, ax=axes[0])
axes[0].set_title('Dispersão')

# Histograma 2D: mostra a densidade quando há muitos pontos sobrepostos
sns.histplot(data=df, x='coluna_num', y='coluna_num_2', bins=30, cmap='viridis', cbar=True, ax=axes[1])
axes[1].set_title('Histograma 2D (densidade)')
plt.show()

In [ ]:
dados_corr = df[['coluna_num', 'coluna_num_2']].dropna()   # scipy não aceita NaN

r, p_r = stats.pearsonr(dados_corr['coluna_num'], dados_corr['coluna_num_2'])
rho, p_rho = stats.spearmanr(dados_corr['coluna_num'], dados_corr['coluna_num_2'])

print(f'Pearson : r = {r:.3f} (p = {p_r:.3g})')
print(f'Spearman: ρ = {rho:.3f} (p = {p_rho:.3g})')

⚠️ **O que observar**
- **Correlação não implica causalidade.** Duas variáveis podem estar correlacionadas por causa de uma terceira, ou por coincidência.
- Pearson **subestima** relações não lineares. Por exemplo, $y = x^5$ é perfeitamente crescente, mas tem r < 1. Spearman = 1 nesse caso.
- Pearson é **sensível a outliers**: poucos pontos extremos criam ou destroem uma correlação. Compare com Spearman e olhe o gráfico.
- O p-valor testa se a correlação é diferente de zero. Com amostras grandes, até correlações fracas e irrelevantes dão p < 0,05.
- Use `alpha` (transparência) em gráficos de dispersão com muitos pontos.

### Matriz de correlação

**O que faz:** calcula a correlação entre todos os pares de colunas numéricas e a mostra como mapa de calor.

**Quando usar:** para triar rapidamente quais pares de variáveis merecem investigação.

In [ ]:
cols_corr = ['coluna_num', 'coluna_num_2', 'coluna_valor', 'coluna_binaria', 'log_coluna_num']

fig, axes = plt.subplots(ncols=2, figsize=(16, 6))
for ax, metodo in zip(axes, ['pearson', 'spearman']):
    corr = df[cols_corr].corr(method=metodo)
    mascara = np.triu(np.ones_like(corr, dtype=bool))     # esconde a metade repetida
    sns.heatmap(corr, mask=mascara, annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, center=0, square=True, ax=ax)
    ax.set_title(f'Correlação de {metodo.capitalize()}')
plt.tight_layout()
plt.show()

⚠️ Use uma paleta **divergente** (`'RdBu_r'`, `'coolwarm'`) com `vmin=-1`, `vmax=1` e `center=0`, para que a cor tenha o mesmo significado em qualquer gráfico. `df.corr(numeric_only=True)` ignora as colunas de texto automaticamente.

## 8.2 Distribuição e relação juntas: `jointplot`

**O que faz:** mostra a dispersão de duas variáveis no centro e a distribuição de cada uma nas margens.

**Quando usar:** para investigar a fundo **um par** de variáveis. Para vários pares, use o `pairplot` (Seção 8.6).

In [ ]:
g = sns.jointplot(data=df, x='coluna_num', y='coluna_num_2', kind='hex', height=6)
# kind: 'scatter' | 'hex' (muitos pontos) | 'kde' | 'hist' | 'reg' (com regressão) | 'resid'
plt.show()

In [ ]:
# Com grupos (hue) e camadas extras sobre o mesmo grid
g = sns.jointplot(data=df, x='log_coluna_num', y='coluna_num_2', hue='coluna_cat_2',
                  kind='kde', height=6, marginal_kws={'common_norm': False})
g.plot_joint(sns.scatterplot, alpha=0.3, s=15)      # adiciona pontos ao gráfico central
plt.show()

## 8.3 Numérica × Categórica: comparar grupos

### Resumo numérico por grupo

**O que faz:** calcula várias métricas por categoria em uma única tabela, com **agregação nomeada**.

**Quando usar:** é a base de qualquer comparação entre segmentos (ranking de categorias, total, média, contagem).

In [ ]:
resumo_grupos = (df
    .groupby('coluna_cat')
    .agg(
        n=('coluna_id', 'count'),
        total=('coluna_valor', 'sum'),
        media=('coluna_num_2', 'mean'),
        mediana=('coluna_num_2', 'median'),
        desvio=('coluna_num_2', 'std'),
        categoria_mais_frequente=('coluna_cat_2', lambda s: s.mode().iloc[0] if not s.mode().empty else None),
    )
    .assign(pct_registros=lambda t: t['n'] / t['n'].sum() * 100)
    .sort_values('total', ascending=False)
    .reset_index())

resumo_grupos

In [ ]:
# Agrupamento por duas variáveis: média e mediana para cada combinação
df.groupby(['coluna_cat', 'coluna_cat_2'])['coluna_num_2'].agg(['mean', 'median']).round(2)

⚠️ **O que observar**
- Sempre mostre o **n de cada grupo**. Médias de grupos pequenos são instáveis.
- Rankings por **soma** favorecem grupos grandes, e rankings por **média** podem ser dominados por grupos minúsculos. Use os dois e filtre por um `n` mínimo.
- Uma média diferente entre grupos não prova diferença real. Para isso, use um teste estatístico (Seção 12).

### Gráficos de comparação

**O que faz:** coloca a distribuição de cada grupo lado a lado (boxplot) e sobreposta (KDE).

**Quando usar:** é a função-padrão para comparar uma variável numérica entre as categorias de outra.

In [ ]:
def comparar_distribuicao(data, var_num, var_cat, ordem=None):
    # Boxplot por grupo + KDEs sobrepostos (normalizados por grupo).
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(15, 5))
    sns.boxplot(data=data, x=var_cat, y=var_num, order=ordem, ax=ax1)
    ax1.set_title(f'{var_num} por {var_cat}')
    sns.kdeplot(data=data, x=var_num, hue=var_cat, hue_order=ordem,
                common_norm=False, fill=True, alpha=0.3, ax=ax2)
    ax2.set_title(f'Distribuição de {var_num} por {var_cat}')
    plt.tight_layout()
    plt.show()

comparar_distribuicao(df, 'coluna_num', 'coluna_cat', ordem=['A', 'B', 'C'])

In [ ]:
# Histogramas sobrepostos: compare formas com stat='density' + common_norm=False
fig, axes = plt.subplots(ncols=2, figsize=(15, 4), sharex=True)
sns.histplot(data=df, x='coluna_num', hue='coluna_cat_2', binwidth=1, kde=True, ax=axes[0])
axes[0].set_title('Contagem (altura depende do tamanho do grupo)')
sns.histplot(data=df, x='coluna_num', hue='coluna_cat_2', binwidth=1, kde=True,
             stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Densidade por grupo (formas comparáveis)')
plt.show()

⚠️ Com `hue`, a contagem absoluta faz grupos maiores parecerem "mais altos". Para comparar **formas**, use `stat='density'` e `common_norm=False`, que normalizam cada grupo separadamente. Sem isso, a normalização é feita sobre o total e os grupos menores continuam baixos.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(15, 5))

# Violino dividido: 2 categorias de hue, cada metade do violino é uma delas
sns.violinplot(data=df, x='coluna_cat', y='coluna_num', hue='coluna_cat_2',
               split=True, inner='quartile', order=['A', 'B', 'C'], ax=axes[0])
axes[0].set_title('Violino dividido por uma segunda categoria')

# Violino + pontos individuais
sns.violinplot(data=df, x='coluna_cat', y='coluna_num', order=['A', 'B', 'C'],
               inner=None, color='lightgray', ax=axes[1])
sns.stripplot(data=df, x='coluna_cat', y='coluna_num', order=['A', 'B', 'C'],
              alpha=0.3, size=2.5, jitter=0.25, ax=axes[1])
axes[1].set_title('Violino + pontos')
plt.show()

In [ ]:
# Barras com a média e a dispersão (o estimador pode ser outra função: np.median, np.std, sum...)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=df, x='coluna_cat', y='coluna_num_2', hue='coluna_cat', order=['A', 'B', 'C'],
            estimator='mean', errorbar='sd', capsize=0.1, palette='viridis', legend=False, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', label_type='center', color='white', fontsize=12)
ax.set_title('Média de coluna_num_2 por grupo (barra de erro = desvio padrão)')
plt.show()

⚠️ **O que observar**
- `errorbar='sd'` mostra a **dispersão dos dados**. `errorbar=('ci', 95)` (padrão) mostra a **incerteza da média**. Deixe claro no gráfico qual das duas foi usada.
- Barras de média escondem a distribuição. Combine com boxplot ou violino, ou mostre os pontos.
- Em `stripplot`/`swarmplot`/`boxplot` com `hue`, `dodge=True` coloca os subgrupos **lado a lado** em vez de sobrepostos.
- A paleta de cores via `palette` exige `hue` no seaborn ≥ 0.13. Use `hue=` igual a `x=` e `legend=False`.

## 8.4 Categórica × Categórica: tabela cruzada

**O que faz:** conta as combinações de duas categóricas. Com `normalize`, mostra a **proporção dentro de cada linha** (ou coluna).

**Quando usar:** para perguntas como "a proporção de Y muda conforme X?".

In [ ]:
pd.crosstab(df['coluna_cat'], df['coluna_cat_2'], margins=True)            # contagens + totais

In [ ]:
tab_pct = pd.crosstab(df['coluna_cat'], df['coluna_binaria'], normalize='index') * 100   # % por linha

fig, axes = plt.subplots(ncols=2, figsize=(14, 4))
sns.heatmap(tab_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[0])
axes[0].set_title('% de coluna_binaria dentro de cada categoria')
tab_pct.plot(kind='barh', stacked=True, ax=axes[1])                       # barras 100% empilhadas
axes[1].set_xlabel('%')
axes[1].legend(title='coluna_binaria', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

⚠️ Escolha a normalização conforme a pergunta. `normalize='index'` responde "dentro de cada X, qual a distribuição de Y?". `normalize='columns'` responde o inverso. O teste de associação (qui-quadrado) está na Seção 12.7.

## 8.5 Segmentação: a relação muda entre grupos?

**O que faz:** repete a análise de relação dentro de cada segmento.

**Quando usar:** quando o gráfico geral parece "misturar" padrões. Uma correlação geral pode desaparecer, ou até se inverter, dentro dos grupos, e grupos diferentes podem ter relações diferentes.

In [ ]:
# Correlação dentro de cada grupo
corr_por_grupo = (df.dropna(subset=['coluna_num', 'coluna_num_2'])
                    .groupby('coluna_cat')[['coluna_num', 'coluna_num_2']]
                    .apply(lambda g: pd.Series({
                        'n': len(g),
                        'pearson': g['coluna_num'].corr(g['coluna_num_2']),
                        'spearman': g['coluna_num'].corr(g['coluna_num_2'], method='spearman'),
                    }))
                    .sort_values('pearson', ascending=False))
corr_por_grupo

In [ ]:
# Um gráfico por grupo (figure-level: cria a própria figura; col_wrap quebra em linhas)
g = sns.relplot(data=df, x='coluna_num', y='coluna_num_2', col='coluna_cat', hue='coluna_cat',
                col_order=['A', 'B', 'C'], col_wrap=3, height=4, alpha=0.5, legend=False)
plt.show()

⚠️ **O que observar**
- Quando a dispersão mostra **duas "nuvens"** separadas (por exemplo, um grupo de valores muito altos), divida os dados por um ponto de corte ou por uma categoria e analise cada parte separadamente. As correlações podem ser bem diferentes em cada uma.
- Muitos pontos sobrepostos de vários grupos em um só gráfico (`hue`) são difíceis de ler. Facetas (`col`) costumam ser mais claras.
- Confira o **n de cada segmento** antes de comparar correlações.

## 8.6 Visão multivariada

### Pairplot

**O que faz:** cria uma matriz de gráficos com a dispersão de cada par de variáveis fora da diagonal e a distribuição de cada variável na diagonal.

**Quando usar:** como triagem visual de **até ~5 colunas numéricas**. Com mais do que isso, o gráfico fica lento e ilegível.

In [ ]:
sns.pairplot(data=df[['coluna_num', 'coluna_num_2', 'log_coluna_num', 'coluna_cat_2']].dropna(),
             hue='coluna_cat_2', diag_kind='kde', corner=True,
             plot_kws={'alpha': 0.4, 's': 15}, diag_kws={'common_norm': False}, height=2.5)
plt.show()
# kind='hist' troca a dispersão por histogramas 2D (melhor com muitos pontos)

### Tabela dinâmica + heatmap

**O que faz:** `pivot_table` cria uma matriz (linhas × colunas × valor agregado), que o heatmap colore.

**Quando usar:** para mostrar uma métrica por duas dimensões ao mesmo tempo (ex.: mês × ano, região × categoria).

In [ ]:
pivot = df.pivot_table(values='coluna_num_2', index='coluna_cat', columns='coluna_ordinal',
                       aggfunc='mean', observed=True)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='magma', linewidths=1, linecolor='white', ax=ax)
ax.set_title('Média de coluna_num_2 por coluna_cat × coluna_ordinal')
plt.show()

🔹 **Alternativa: `sns.clustermap(pivot, standard_scale=1, cmap='coolwarm')`** reordena linhas e colunas por **similaridade** (agrupamento hierárquico), o que revela quais categorias se comportam de forma parecida. `standard_scale=1` normaliza cada coluna para 0-1, permitindo comparar colunas com escalas diferentes.

---
# 9. Análise temporal

## 9.1 Componentes de data e duração

**O que faz:** extrai partes da data (ano, mês, dia da semana) e calcula intervalos entre duas datas.

**Quando usar:** sempre que houver colunas de data. Elas precisam estar como `datetime64` (conversão na Seção 4.2).

In [ ]:
df['ano'] = df['coluna_data'].dt.year
df['mes'] = df['coluna_data'].dt.month                  # número: ordena corretamente
df['mes_nome'] = df['coluna_data'].dt.month_name()      # nome: bom para rótulos
df['dia_semana'] = df['coluna_data'].dt.day_name()
df['ano_mes'] = df['coluna_data'].dt.to_period('M')     # período mensal (ex.: 2024-03)

df['duracao'] = df['coluna_data_fim'] - df['coluna_data']   # Timedelta
df['duracao_dias'] = df['duracao'].dt.days                  # número de dias (int), para cálculos e gráficos

df[['coluna_data', 'coluna_data_fim', 'ano', 'mes', 'mes_nome', 'dia_semana', 'ano_mes', 'duracao_dias']].head()

⚠️ **O que observar**
- Agrupar por `mes_nome` ordena os meses **alfabeticamente** (April, August, ...). Agrupe por `mes` ou `ano_mes` e use o nome só como rótulo.
- `dt.month_name(locale='pt_BR.UTF-8')` retorna os nomes em português, se o sistema tiver esse locale instalado.
- Durações negativas indicam datas invertidas ou erro de registro.
- Para médias de duração, use `duracao_dias` (número). A média de um `Timedelta` funciona, mas é menos legível.

## 9.2 Evolução no tempo

**O que faz:** agrega a métrica por período (contagem, soma, média) e plota a série.

**Quando usar:** para identificar tendência, sazonalidade, picos e quedas.

In [ ]:
serie_mensal = (df
    .groupby('ano_mes')
    .agg(n=('coluna_id', 'count'), total=('coluna_valor', 'sum'), media=('coluna_num_2', 'mean')))
serie_mensal['media_movel_3m'] = serie_mensal['total'].rolling(3).mean()   # suaviza oscilações

fig, ax = plt.subplots(figsize=(12, 4))
serie_mensal[['total', 'media_movel_3m']].plot(ax=ax, marker='o')
ax.set_title('Total mensal de coluna_valor')
ax.set_xlabel('')
plt.show()

In [ ]:
# Alternativa com a data como índice: resample agrupa por frequência ('D', 'W', 'MS', 'QS', 'YS')
df.set_index('coluna_data')['coluna_valor'].resample('MS').sum().head()

## 9.3 Sazonalidade: período × categoria

**O que faz:** cruza o período (mês, dia da semana) com outra dimensão em um heatmap.

**Quando usar:** para encontrar padrões recorrentes (meses de pico, dias fracos).

In [ ]:
ordem_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot_tempo = (df.pivot_table(values='coluna_id', index='dia_semana', columns='mes', aggfunc='count')
                 .reindex(ordem_dias))

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_tempo, cmap='YlGnBu', annot=True, fmt='.0f', ax=ax)
ax.set_title('Registros por dia da semana × mês')
plt.show()

⚠️ **O que observar**
- Períodos incompletos (o mês atual, o primeiro mês da base) parecem quedas, mas não são. Confira as datas mínima e máxima: `df['coluna_data'].agg(['min', 'max'])`.
- Compare períodos equivalentes (mês contra o mesmo mês do ano anterior) quando houver sazonalidade.

## 9.4 Variação entre períodos ➕

**O que faz:** compara cada período com o anterior (**mês a mês**, MoM), com o mesmo período do ano anterior (**ano a ano**, YoY) e calcula o **acumulado no ano** (YTD).

**Quando usar:** em qualquer acompanhamento de indicador. A variação diz se o resultado melhorou ou piorou, e o valor absoluto sozinho não diz.

| Medida | Código | Pergunta |
|---|---|---|
| Variação % sobre o período anterior | `.pct_change()` | cresceu em relação ao mês passado? |
| Variação % sobre 12 períodos antes | `.pct_change(12)` | cresceu em relação ao mesmo mês do ano anterior? |
| Diferença absoluta | `.diff()` | quanto mudou, na unidade original? |
| Valor do período anterior | `.shift()` | base para cálculos próprios |
| Acumulado | `.cumsum()` (por ano: `groupby(ano).cumsum()`) | quanto já foi realizado no ano? |

In [ ]:
receita_mensal = (eventos[eventos['coluna_etapa'] == 'etapa_3']
                  .set_index('coluna_data_evento')['coluna_valor_pedido']
                  .resample('MS').sum()               # garante todos os meses, inclusive os sem venda
                  .to_frame('receita'))

receita_mensal['var_mom'] = receita_mensal['receita'].pct_change()
receita_mensal['var_yoy'] = receita_mensal['receita'].pct_change(12)
receita_mensal['dif_mom'] = receita_mensal['receita'].diff()
receita_mensal['acumulado_ano'] = receita_mensal.groupby(receita_mensal.index.year)['receita'].cumsum()

receita_mensal.tail(12)   # var_mom e var_yoy em fração: 0.05 = +5%

In [ ]:
# Comparativo ano contra ano: meses nas linhas, anos nas colunas
comparativo = receita_mensal.pivot_table(index=receita_mensal.index.month,
                                         columns=receita_mensal.index.year, values='receita')
comparativo.index.name = 'mes'
ano_atual, ano_anterior = comparativo.columns.max(), comparativo.columns.max() - 1
comparativo['var_yoy'] = comparativo[ano_atual] / comparativo[ano_anterior] - 1

fig, axes = plt.subplots(ncols=2, figsize=(15, 4))
comparativo[[ano_anterior, ano_atual]].plot(ax=axes[0], marker='o')
axes[0].set_title('Receita mensal: ano atual × ano anterior')
axes[0].legend(title='ano')
receita_mensal['acumulado_ano'].plot(ax=axes[1])
axes[1].set_title('Receita acumulada no ano (reinicia em janeiro)')
plt.show()

⚠️ **O que observar**
- `pct_change(12)` só equivale a "ano a ano" se a série tiver **todos os meses**, sem lacunas. Por isso a série foi montada com `resample('MS')`.
- Variações percentuais sobre bases pequenas parecem enormes (de 10 para 30 = +200%). Reporte sempre o valor absoluto junto da variação.
- Não compare um período **incompleto** (mês corrente) com um completo. Para o ano corrente, compare o acumulado até o mesmo mês do ano anterior.
- O início de uma base (por exemplo, o primeiro ano de operação) costuma mostrar crescimento artificial. Com sazonalidade, prefira YoY a MoM.

---
# 10. Métricas de negócio ➕

**KPIs** (indicadores-chave de desempenho) traduzem os dados em respostas para o negócio: quanto vendemos, quantos convertem, quanto vale cada cliente, quantos voltam.

> Antes de calcular, **defina cada KPI por escrito**: numerador, denominador, período, unidade e filtros. Duas pessoas calculando "taxa de conversão" com denominadores diferentes chegam a números diferentes e ambos parecem certos.

| KPI | Fórmula | Pergunta |
|---|---|---|
| Taxa de conversão | clientes que compraram / clientes que entraram | qual fração avança até a compra? |
| Receita | soma do valor dos pedidos | quanto foi vendido? |
| Ticket médio | receita / número de pedidos | quanto vale cada pedido? |
| Receita por cliente | receita / clientes | quanto vale cada cliente que entrou? |
| Pedidos por comprador (frequência) | pedidos / clientes que compraram | quantas vezes cada comprador compra? |
| Taxa de retenção | clientes ativos no período *k* / clientes da coorte | quantos continuam comprando? (churn = 1 - retenção) |
| Atingimento de meta | realizado / meta | quanto da meta foi cumprido? |

## 10.1 Base no nível de cliente

**O que faz:** transforma a tabela de eventos (uma linha por evento) em uma tabela de clientes (uma linha por cliente), com indicadores de conversão, pedidos e receita.

**Quando usar:** quase todo KPI de cliente (conversão, frequência, retenção, teste A/B) parte dessa tabela. Definir a **granularidade** certa (evento, pedido ou cliente) é o primeiro passo.

In [ ]:
pedidos = eventos[eventos['coluna_etapa'] == 'etapa_3']

clientes = (eventos
    .groupby('coluna_id_cliente')
    .agg(coluna_variante=('coluna_variante', 'first'),
         data_entrada=('coluna_data_evento', 'min'),
         converteu=('coluna_etapa', lambda s: int((s == 'etapa_3').any()))))

resumo_pedidos = pedidos.groupby('coluna_id_cliente').agg(
    n_pedidos=('coluna_valor_pedido', 'count'),
    receita=('coluna_valor_pedido', 'sum'),
    primeira_compra=('coluna_data_evento', 'min'))

clientes = clientes.join(resumo_pedidos)                       # junção pelo índice (coluna_id_cliente)
clientes[['n_pedidos', 'receita']] = clientes[['n_pedidos', 'receita']].fillna(0)   # quem não comprou = 0
clientes.head()

## 10.2 KPIs gerais e por segmento

**O que faz:** calcula o painel de KPIs para a base inteira e para cada segmento com a mesma função, o que garante a mesma definição em todos os recortes.

In [ ]:
def calcular_kpis(clientes, pedidos):
    # Painel de KPIs com definições fixas (ver tabela da Seção 10).
    n_compradores = clientes['converteu'].sum()
    receita = pedidos['coluna_valor_pedido'].sum()
    return pd.Series({
        'clientes': len(clientes),
        'compradores': n_compradores,
        'taxa_conversao': n_compradores / len(clientes),
        'pedidos': len(pedidos),
        'receita': receita,
        'ticket_medio': receita / len(pedidos),
        'receita_por_cliente': receita / len(clientes),
        'pedidos_por_comprador': len(pedidos) / n_compradores,
    })

kpis_por_segmento = pd.DataFrame({
    'total': calcular_kpis(clientes, pedidos),
    **{seg: calcular_kpis(clientes[clientes['coluna_variante'] == seg], pedidos[pedidos['coluna_variante'] == seg])
       for seg in sorted(clientes['coluna_variante'].unique())},
})
kpis_por_segmento.round(3)

In [ ]:
# Atingimento de meta: realizado (tabela principal) × meta (tabela de dimensão, Seção 4.10)
atingimento = (df_com_categoria
    .dropna(subset=['coluna_meta'])
    .groupby('coluna_cat')
    .agg(realizado=('coluna_valor', 'sum'), meta=('coluna_meta', 'first'))
    .assign(atingimento=lambda t: t['realizado'] / t['meta'],
            gap=lambda t: t['realizado'] - t['meta'])
    .sort_values('atingimento', ascending=False))
atingimento

⚠️ **O que observar**
- Calcule taxas a partir das **somas** (total de pedidos / total de clientes), não pela média de taxas de subgrupos, que dá o mesmo peso a grupos pequenos e grandes.
- Ticket médio é uma média: com valores muito assimétricos, reporte também a mediana do pedido.
- Um KPI isolado engana. Receita subindo com conversão caindo pode significar mais tráfego de pior qualidade. Analise os indicadores em conjunto.

## 10.3 Funil de conversão

**O que faz:** conta quantos clientes chegam a cada etapa de um processo sequencial e calcula a conversão entre etapas.

**Quando usar:** em jornadas com etapas (visita → cadastro → compra, lead → proposta → contrato) para descobrir **onde** está a maior perda.

In [ ]:
ordem_etapas = ['etapa_1', 'etapa_2', 'etapa_3']

funil = (eventos.groupby('coluna_etapa')['coluna_id_cliente'].nunique()
                .reindex(ordem_etapas).to_frame('clientes'))
funil['conv_etapa_anterior'] = funil['clientes'] / funil['clientes'].shift(1)
funil['conv_acumulada'] = funil['clientes'] / funil['clientes'].iloc[0]
funil

In [ ]:
funil_segmento = (eventos.pivot_table(index='coluna_etapa', columns='coluna_variante',
                                      values='coluna_id_cliente', aggfunc='nunique')
                         .reindex(ordem_etapas))
funil_segmento_pct = funil_segmento / funil_segmento.iloc[0]   # % da primeira etapa, por segmento

fig, axes = plt.subplots(ncols=2, figsize=(15, 4))
axes[0].barh(funil.index, funil['clientes'], color='#2a9d8f')
axes[0].invert_yaxis()
axes[0].bar_label(axes[0].containers[0],
                  labels=[f'{n:,.0f} ({p:.0%})' for n, p in zip(funil['clientes'], funil['conv_acumulada'])],
                  padding=3)
axes[0].set_title('Funil geral')
funil_segmento_pct.plot(kind='bar', ax=axes[1], rot=0)
axes[1].set_title('Conversão acumulada por segmento')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
plt.tight_layout()
plt.show()

⚠️ Conte **clientes únicos** (`nunique`) em cada etapa, e não eventos, senão quem repete uma etapa infla o funil. Confirme que a ordem das etapas é obrigatória: se for possível pular etapas, a conversão "da etapa anterior" pode passar de 100%.

## 10.4 Análise de coorte e retenção

**O que faz:** agrupa os clientes pela **coorte** (mês da primeira compra) e acompanha, mês a mês, quantos continuam comprando.

**Quando usar:** para medir retenção e churn e para comparar se clientes conquistados em períodos diferentes se comportam de forma diferente. A comparação entre coortes isola o efeito do "tempo de casa".

In [ ]:
pedidos_coorte = pedidos.assign(
    mes_pedido=pedidos['coluna_data_evento'].dt.to_period('M'),
    coorte=pedidos.groupby('coluna_id_cliente')['coluna_data_evento'].transform('min').dt.to_period('M'),
)
pedidos_coorte['periodo'] = ((pedidos_coorte['mes_pedido'].dt.year - pedidos_coorte['coorte'].dt.year) * 12
                             + (pedidos_coorte['mes_pedido'].dt.month - pedidos_coorte['coorte'].dt.month))

matriz_coorte = pedidos_coorte.pivot_table(index='coorte', columns='periodo',
                                           values='coluna_id_cliente', aggfunc='nunique')
retencao = matriz_coorte.div(matriz_coorte[0], axis=0)   # período 0 = 100% da coorte

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(retencao.iloc[:12, :7], annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=1, ax=ax)
ax.set_title('Retenção por coorte (meses desde a primeira compra)')
ax.set_xlabel('Meses desde a primeira compra')
ax.set_ylabel('Coorte (mês da primeira compra)')
plt.show()

In [ ]:
# Curva média de retenção, ponderada pelo tamanho das coortes, usando só coortes que já viveram cada período
ultimo_mes = pedidos_coorte['mes_pedido'].max()
idade_coorte = pd.Series([(ultimo_mes - c).n for c in matriz_coorte.index], index=matriz_coorte.index)
observado = pd.DataFrame({k: idade_coorte >= k for k in matriz_coorte.columns})

retencao_media = (matriz_coorte.fillna(0).where(observado).sum()
                  / observado.mul(matriz_coorte[0], axis=0).sum())
retencao_media.iloc[:7].to_frame('retencao_media').assign(churn_acumulado=lambda t: 1 - t['retencao_media'])

⚠️ **O que observar**
- Coortes recentes ainda não tiveram tempo de chegar aos períodos longos (a parte vazia da matriz). Não as trate como retenção zero.
- Coortes pequenas oscilam muito. Mostre o tamanho de cada coorte (`matriz_coorte[0]`) junto do heatmap.
- Defina "ativo" de acordo com o ciclo do negócio. Uma compra por mês faz sentido para consumo recorrente, mas não para bens duráveis.

---
# 11. Visualização: referência técnica

As seções anteriores mostram **qual gráfico usar em cada análise**. Esta seção reúne **como construir, combinar e formatar** gráficos.

## 11.1 Qual gráfico usar?

| Objetivo | Gráfico | Função seaborn / matplotlib |
|---|---|---|
| Distribuição de 1 numérica | histograma, KDE, boxplot, violino | `histplot`, `kdeplot`, `boxplot`, `violinplot` |
| Frequência de categorias | barras (horizontais se rótulos longos) | `countplot`, `barplot`, `ax.barh` |
| Comparar numérica entre grupos | boxplot, violino, strip/swarm, barras com erro | `boxplot`, `violinplot`, `stripplot`, `swarmplot`, `barplot` |
| Relação entre 2 numéricas | dispersão, hexbin, histograma 2D, regressão | `scatterplot`, `jointplot`, `histplot(x, y)`, `regplot` |
| Relação entre várias numéricas | pairplot, heatmap de correlação | `pairplot`, `heatmap` |
| Métrica por 2 categorias | heatmap de tabela dinâmica | `pivot_table` + `heatmap` |
| Evolução no tempo | linhas | `lineplot`, `Series.plot()` |
| Proporções entre grupos | barras 100% empilhadas | `crosstab(normalize=...).plot(kind='barh', stacked=True)` |

## 11.2 Estrutura matplotlib: figura, eixos e formatação

**O que faz:** `fig, ax = plt.subplots()` cria uma figura (`fig`) e um ou mais eixos (`ax`). Funções seaborn *axes-level* desenham no eixo indicado por `ax=`.

**Quando usar:** é o padrão de todos os gráficos, porque dá controle total sobre tamanho, título, eixos e grade de vários gráficos.

In [ ]:
# Grade de gráficos: percorra os eixos com axes.flat
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 7), sharex=False)
colunas = ['coluna_num', 'coluna_num_2', 'coluna_valor', 'duracao_dias']

for ax, coluna in zip(axes.flat, colunas):
    sns.histplot(data=df, x=coluna, ax=ax, kde=True)
    ax.axvline(df[coluna].median(), color='red', linestyle='--', label='mediana')  # linha de referência
    ax.set_title(coluna)
    ax.legend()

fig.suptitle('Distribuição das variáveis numéricas', fontsize=14)
plt.tight_layout()       # evita sobreposição de títulos e rótulos
plt.show()

### Gráfico de apresentação (pronto para relatório)

Ranking em barras horizontais, ordenado, com rótulos, fonte dos dados e exportação.

In [ ]:
ranking = df['coluna_cat'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(ranking.index, ranking.values, color='#2a9d8f')
ax.invert_yaxis()                                     # maior no topo
ax.bar_label(ax.containers[0], fmt='%d', padding=3)
ax.set_xlabel('Número de registros')
ax.set_title(f'Registros por categoria ({PERIODO})', loc='left', fontsize=14)
ax.grid(axis='x', color='gray', linestyle='--', linewidth=0.5)
ax.grid(axis='y', visible=False)
sns.despine(left=True)
fig.text(0.99, 0.01, 'Fonte: nome_da_fonte', ha='right', fontsize=8, color='gray')

PASTA_SAIDA.mkdir(exist_ok=True)
fig.savefig(CAMINHO_SAIDA_GRAFICO, dpi=150, bbox_inches='tight')   # bbox_inches evita cortar rótulos
plt.show()

**Formatação frequente**

| Tarefa | Código |
|---|---|
| Título da figura / do eixo | `fig.suptitle('...')` / `ax.set_title('...')` |
| Rótulos dos eixos | `ax.set_xlabel('...')`, `ax.set_ylabel('...')` |
| Girar rótulos | `plt.xticks(rotation=45, ha='right')` ou `ax.tick_params(axis='x', rotation=45)` |
| Renomear rótulos de categorias | `ax.set_xticks(ax.get_xticks(), ['Rótulo 1', 'Rótulo 2'])` |
| Limites dos eixos (zoom) | `ax.set_xlim(0, 100)`, `ax.set_ylim(0, 50_000)` |
| Escala log | `ax.set_xscale('log')` |
| Linhas de referência | `ax.axvline(x)`, `ax.axhline(y)` |
| Texto / nota de fonte | `ax.text(x, y, '...')`, `fig.text(x, y, '...')` |
| Valores nas barras | `ax.bar_label(ax.containers[0], fmt='%.1f')` |
| Legenda fora do gráfico | `ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')` |
| Tamanho do marcador / transparência | `s=20`, `alpha=0.5` |
| Gráfico rápido direto do pandas | `df.plot(x='coluna_1', y='coluna_2', kind='bar')` |

## 11.3 Funções *axes-level* × *figure-level* e grades

| Axes-level (aceita `ax=`) | Figure-level (cria a própria figura, aceita `col=`/`row=`) |
|---|---|
| `histplot`, `kdeplot`, `ecdfplot` | `displot` |
| `scatterplot`, `lineplot` | `relplot` |
| `boxplot`, `violinplot`, `stripplot`, `swarmplot`, `barplot`, `countplot` | `catplot(kind=...)` |
| `regplot` | `lmplot` |
| - | `pairplot`, `jointplot` |

⚠️ Funções *figure-level* **não aceitam `ax=`** e retornam um objeto de grade (`FacetGrid`, `PairGrid`, `JointGrid`). O tamanho é controlado por `height` (altura de cada painel) e `aspect` (largura = height × aspect), não por `figsize`.

In [ ]:
# catplot: qualquer gráfico categórico, facetado por outra variável
g = sns.catplot(data=df, x='coluna_cat', y='coluna_num', col='coluna_cat_2', kind='box',
                order=['A', 'B', 'C'], height=4, aspect=1.1)
g.set_titles('coluna_cat_2 = {col_name}')
plt.show()

In [ ]:
# FacetGrid: aplica qualquer função de gráfico a cada combinação de linha/coluna
g = sns.FacetGrid(df, col='coluna_cat_2', row='nivel', hue='coluna_binaria', height=3, aspect=1.4)
g.map_dataframe(sns.scatterplot, x='coluna_num', y='coluna_num_2', alpha=0.5)
g.add_legend()
plt.show()

In [ ]:
# PairGrid: um tipo de gráfico diferente na diagonal, acima e abaixo dela
g = sns.PairGrid(df[['coluna_num', 'coluna_num_2', 'log_coluna_num']].dropna(), height=2.5)
g.map_diag(sns.histplot)
g.map_upper(sns.scatterplot, alpha=0.3, s=10)
g.map_lower(sns.kdeplot, fill=True)
plt.show()

🔹 **`sns.JointGrid(data=df, x=..., y=...)`** é a versão configurável do `jointplot`: `g.plot(sns.regplot, sns.histplot)` define o gráfico central e o das margens. `g.plot_joint(...)` e `g.plot_marginals(...)` adicionam camadas, como na Seção 8.2.

## 11.4 Estilos, contexto e cores

| Função | Controla | Opções |
|---|---|---|
| `sns.set_theme(style=, context=, palette=)` | tudo de uma vez | ver abaixo |
| `sns.set_style()` | fundo e grade | `'whitegrid'`, `'darkgrid'`, `'white'`, `'dark'`, `'ticks'` |
| `sns.set_context()` | escala de fontes e linhas | `'paper'` < `'notebook'` < `'talk'` < `'poster'` (+ `font_scale=`) |
| `sns.despine()` | remove bordas superior/direita (`left=True` remove a esquerda) | - |
| `palette=` | cores | qualitativa: `'deep'`, `'Set1'`, `'Set2'`; sequencial: `'viridis'`, `'Blues'`, `'magma'`; divergente: `'RdBu_r'`, `'coolwarm'`; lista própria: `['#acacac', '#34ba34']` |

⚠️ **O que observar**
- Escolha o tipo de paleta pelo tipo de dado: **qualitativa** para categorias, **sequencial** para intensidade e **divergente** para valores com um centro (como a correlação, centrada em 0).
- Evite `'rainbow'` / `'jet'`: distorcem a percepção e não são acessíveis para daltônicos.
- Use `hue_order` / `order` para manter cores e ordem **consistentes** entre gráficos.
- Para destacar um grupo, pinte-o com uma cor forte e deixe os demais em cinza (`palette=['#acacac', '#34ba34']`).

## 11.5 Funções e parâmetros desatualizados

Material antigo usa funções que foram removidas ou geram avisos nas versões atuais:

| Antigo | Atual |
|---|---|
| `sns.distplot(x)` | `sns.histplot(x, kde=True)` ou `sns.displot(...)` |
| `sns.distplot(x, kde=False, bins=30)` | `sns.histplot(x, bins=30)` |
| `sns.factorplot(...)` | `sns.catplot(...)` |
| `size=` em `lmplot`/`catplot` | `height=` |
| `violinplot(bw=...)` | `violinplot(bw_adjust=...)` |
| `barplot(..., palette=...)` sem `hue` | `barplot(..., hue=<mesma variável de x>, palette=..., legend=False)` |
| `barplot(ci='sd')` | `barplot(errorbar='sd')` |
| `df.corr()` com colunas de texto | `df.corr(numeric_only=True)` |
| `np.NaN` | `np.nan` |
| `df.select_dtypes('object')` para texto | `df.select_dtypes(exclude='number')`, que funciona no pandas 2 e 3 |

---
# 12. Estatística inferencial

A estatística inferencial responde: **a diferença ou relação observada na amostra é real, ou poderia ser fruto do acaso?**

## 12.1 Conceitos essenciais

| Conceito | Significado prático |
|---|---|
| **H₀** (hipótese nula) | "não há diferença/efeito/relação" (o estado atual) |
| **H₁** (hipótese alternativa) | "há diferença/efeito/relação" |
| **α** (nível de significância) | risco aceito de concluir que há efeito quando não há; padrão **0,05** (zona de "maioria" C = 95%) |
| **p-valor** | probabilidade de observar um resultado tão extremo quanto o obtido **se H₀ fosse verdadeira** |
| **Decisão** | `p < α` → rejeita H₀ (diferença estatisticamente significativa); `p ≥ α` → não rejeita H₀ (isso **não prova** que H₀ é verdadeira) |
| **Erro tipo I** (α) | falso positivo: rejeitar H₀ verdadeira |
| **Erro tipo II** (β) | falso negativo: não rejeitar H₀ falsa |
| **Poder** (1 - β) | chance de detectar um efeito que existe; cresce com o tamanho da amostra |
| **Teorema Central do Limite** | com amostras aleatórias, independentes e grandes (n ≥ ~30), a **média amostral** tem distribuição aproximadamente normal, mesmo que os dados não tenham |

**Amostras independentes × pareadas**
- **Independentes**: indivíduos diferentes em cada grupo (ex.: categoria A × categoria B).
- **Pareadas**: as mesmas unidades medidas duas vezes (ex.: antes × depois).

**Testes paramétricos × não paramétricos**
- **Paramétricos** (Z, t, ANOVA) comparam **médias** e pressupõem normalidade e, em geral, variâncias homogêneas. Têm mais poder quando os pressupostos são atendidos.
- **Não paramétricos** (Mann-Whitney, Kruskal-Wallis) trabalham com **postos (rankings)**. Use-os quando os pressupostos falham: amostras pequenas e não normais, ou outliers fortes.

In [ ]:
# Valor crítico Z para um teste bilateral: |Z| > z_critico cai na região de rejeição (α dividido nas duas caudas)
z_critico = stats.norm.ppf(1 - ALFA / 2)
print(f'α = {ALFA} → Z crítico = ±{z_critico:.2f}')

## 12.2 Qual teste usar?

| Pergunta | Pressupostos OK (paramétrico) | Pressupostos violados (não paramétrico) | Pós-teste (se significativo) |
|---|---|---|---|
| Média de 1 grupo = valor de referência? | t de uma amostra (`ttest_1samp`) | Wilcoxon (`wilcoxon(x - ref)`) | - |
| 2 grupos independentes diferem? | t independente (`ttest_ind`); **Welch** se variâncias diferentes | Mann-Whitney (`mannwhitneyu`) | - |
| 2 medidas pareadas diferem? | t pareado (`ttest_rel`) | Wilcoxon (`wilcoxon`) | - |
| 3+ grupos independentes diferem? | ANOVA one-way (`f_oneway`); **ANOVA de Welch** se variâncias diferentes | Kruskal-Wallis (`kruskal`) | Tukey (ANOVA) · Games-Howell (Welch) · Dunn (Kruskal) |
| 2 categóricas associadas? | Qui-quadrado (`chi2_contingency`) | Exato de Fisher (tabelas 2×2 pequenas) | - |
| 2 proporções (taxas) diferem? Teste A/B | Teste Z de proporções (`proportions_ztest`, Seção 12.8) | Qui-quadrado / Fisher (Seção 12.7) | - |
| 2 numéricas relacionadas? | Pearson (`pearsonr`) | Spearman (`spearmanr`) | - (Seção 8.1) |

**Roteiro de qualquer teste:**
1. Formule H₀ e H₁ e defina α **antes** de olhar o resultado.
2. Verifique os pressupostos: independência, normalidade (12.3) e homogeneidade de variâncias (12.3).
3. Escolha e aplique o teste.
4. Interprete o **p-valor** e o **tamanho de efeito** (12.9).
5. Se houver mais de 2 grupos e o resultado for significativo, aplique o **pós-teste** para saber *quais* grupos diferem.

### Preparação dos grupos

Os testes recebem uma série de valores por grupo. Exclua categorias não informativas e remova os nulos.

In [ ]:
var_num, var_cat = 'coluna_num', 'coluna_cat'          # variável analisada e variável de agrupamento
dados_teste = df[df[var_cat].isin(['A', 'B', 'C'])].dropna(subset=[var_num])

grupos = {nome: g[var_num].to_numpy() for nome, g in dados_teste.groupby(var_cat)}
{nome: len(valores) for nome, valores in grupos.items()}    # n de cada grupo

## 12.3 Verificação de pressupostos

### Normalidade: Shapiro-Wilk
- H₀: os dados vêm de uma distribuição normal. **p > α → normalidade não rejeitada.**
- Teste **cada grupo separadamente**.

### Homogeneidade de variâncias: Levene
- H₀: os grupos têm variâncias iguais. **p > α → variâncias homogêneas.**

In [ ]:
def testar_normalidade(grupos, alfa=ALFA):
    # Shapiro-Wilk + assimetria e curtose por grupo.
    linhas = []
    for nome, valores in grupos.items():
        w, p = stats.shapiro(valores)
        linhas.append({'grupo': nome, 'n': len(valores), 'W': w, 'p_valor': p,
                       'normal (p > α)': p > alfa,
                       'assimetria': stats.skew(valores), 'curtose': stats.kurtosis(valores)})
    return pd.DataFrame(linhas).round(4)

testar_normalidade(grupos)

In [ ]:
estat_lev, p_lev = stats.levene(*grupos.values(), center='median')   # center='median' é robusto a não normalidade
print(f'Levene: W = {estat_lev:.3f}, p = {p_lev:.4f} → variâncias {"homogêneas" if p_lev > ALFA else "diferentes"}')

⚠️ **O que observar**
- O Shapiro-Wilk é **muito sensível com amostras grandes** (acima de alguns milhares de registros): rejeita normalidade por desvios mínimos e irrelevantes. Com **n grande**, apoie-se também em **assimetria/curtose, QQ-plot (Seção 6.4) e no Teorema Central do Limite**. Com |assimetria| < 1 e n ≥ 30 por grupo, testes t e ANOVA (preferencialmente na versão de Welch) costumam ser robustos.
- Com **n pequeno** (< 30), o Shapiro tem pouco poder. Não rejeitar a normalidade não garante que ela exista.
- **Independência das observações** não se testa: ela vem do desenho da coleta (sem repetição do mesmo indivíduo em grupos diferentes, por exemplo).
- Na dúvida entre Student e Welch, use **Welch**. Ele perde pouco quando as variâncias são iguais e protege quando não são.

## 12.4 Um grupo contra um valor de referência

**Quando usar:** "a média do grupo é diferente de um valor-alvo/histórico?"

- **Teste t de uma amostra**: o desvio padrão populacional é desconhecido (caso comum).
- **Teste Z**: só quando o desvio padrão **populacional** é conhecido, o que é raro na prática. Para n grande, os resultados de t e Z convergem.

In [ ]:
valor_referencia = 10
amostra = dados_teste[var_num]

t_stat, p = stats.ttest_1samp(amostra, popmean=valor_referencia)
print(f'Teste t (1 amostra): t = {t_stat:.3f}, p = {p:.4g}')

# Teste Z (statsmodels), para amostras grandes
from statsmodels.stats.weightstats import ztest
z_stat, p_z = ztest(amostra, value=valor_referencia)
print(f'Teste Z            : Z = {z_stat:.3f}, p = {p_z:.4g}')

## 12.5 Dois grupos

### Independentes: teste t (Student/Welch) e Mann-Whitney

In [ ]:
grupo_1 = df.loc[df['coluna_binaria'] == 0, 'coluna_num_2'].dropna()
grupo_2 = df.loc[df['coluna_binaria'] == 1, 'coluna_num_2'].dropna()

# Paramétrico: equal_var=True → Student | equal_var=False → Welch (recomendado como padrão)
t_stat, p_t = stats.ttest_ind(grupo_1, grupo_2, equal_var=False)
print(f'Teste t de Welch : t = {t_stat:.3f}, p = {p_t:.4g}')

# Não paramétrico: compara as distribuições (postos), não as médias
u_stat, p_u = stats.mannwhitneyu(grupo_1, grupo_2, alternative='two-sided')
print(f'Mann-Whitney     : U = {u_stat:.0f}, p = {p_u:.4g}')

### Pareados (mesmas unidades medidas duas vezes)

```python
stats.ttest_rel(df['coluna_antes'], df['coluna_depois'])    # paramétrico
stats.wilcoxon(df['coluna_antes'], df['coluna_depois'])     # não paramétrico
```

⚠️ **O que observar**
- `alternative='two-sided'` (padrão) testa "diferente". Use `'less'` ou `'greater'` apenas se a direção foi definida **antes** de ver os dados.
- Mann-Whitney com distribuições de formas diferentes testa diferenças de **distribuição**, não só de mediana. Reporte medianas junto com o resultado.

## 12.6 Três ou mais grupos

**Por que não fazer vários testes t?** Cada teste tem 5% de chance de falso positivo, e esses erros se acumulam. A ANOVA e o Kruskal-Wallis testam todos os grupos de uma vez. O pós-teste corrige as comparações múltiplas.

### Paramétrico: ANOVA (e ANOVA de Welch)
- Estatística $F = \frac{\text{variância entre grupos}}{\text{variância dentro dos grupos}}$. Um F alto indica que os grupos diferem mais do que o esperado pelo acaso.
- A ANOVA diz **se** há diferença, mas não **onde**. Para isso, use o pós-teste.

In [ ]:
f_stat, p_anova = stats.f_oneway(*grupos.values())                       # ANOVA clássica (variâncias iguais)
f_w, p_welch = stats.f_oneway(*grupos.values(), equal_var=False)         # ANOVA de Welch (scipy ≥ 1.15)
print(f'ANOVA one-way: F = {f_stat:.2f}, p = {p_anova:.4g}')
print(f'ANOVA Welch  : F = {f_w:.2f}, p = {p_welch:.4g}')

In [ ]:
# Pós-teste de Tukey (após ANOVA significativa): quais pares diferem?
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=dados_teste[var_num], groups=dados_teste[var_cat], alpha=ALFA)
print(tukey.summary())     # reject=True → diferença significativa entre o par
# alternativa scipy: stats.tukey_hsd(*grupos.values())

### Não paramétrico: Kruskal-Wallis + Dunn

In [ ]:
h_stat, p_kw = stats.kruskal(*grupos.values())
print(f'Kruskal-Wallis: H = {h_stat:.2f}, p = {p_kw:.4g}')

In [ ]:
# Pós-teste de Dunn (após Kruskal significativo), com correção para comparações múltiplas
import scikit_posthocs as sp

sp.posthoc_dunn(dados_teste, val_col=var_num, group_col=var_cat, p_adjust='bonferroni').round(4)
# matriz de p-valores: célula < α → o par difere

⚠️ **O que observar**
- Só aplique o pós-teste se o teste global (ANOVA ou Kruskal) for significativo.
- O pós-teste deve combinar com o teste global: **Tukey** após ANOVA com variâncias iguais, **Games-Howell** após ANOVA de Welch (Seção 12.11) e **Dunn** após Kruskal-Wallis.
- *Two-way* ANOVA (dois fatores ao mesmo tempo) requer `statsmodels` (`ols` + `anova_lm`) ou `pingouin.anova(between=[fator_1, fator_2])`.

## 12.7 Associação entre duas categóricas: qui-quadrado ➕

**Quando usar:** "a distribuição de uma categórica depende da outra?" (Seção 8.4).

In [ ]:
tabela = pd.crosstab(df['coluna_cat'], df['coluna_binaria'])
chi2, p_chi2, gl, esperado = stats.chi2_contingency(tabela)
print(f'Qui-quadrado: χ² = {chi2:.2f}, gl = {gl}, p = {p_chi2:.4g}')

⚠️ O teste exige frequências esperadas ≥ 5 na maioria das células (confira em `esperado`). Se não for o caso, agrupe categorias, ou use `stats.fisher_exact` em tabelas 2×2.

## 12.8 Comparação de proporções e testes A/B ➕

**O que faz:** testa se duas **taxas** (conversão, churn, inadimplência) são diferentes e calcula **quantas observações** são necessárias para detectar uma diferença relevante.

**Quando usar:** em experimentos A/B (variante de controle × variante nova) e em qualquer comparação de percentuais entre dois grupos.

**Roteiro de um teste A/B**
1. Defina **antes**: a métrica principal, o efeito mínimo que vale detectar (MDE), α e o poder (em geral 80%).
2. Calcule o **tamanho de amostra** e a duração necessária.
3. Distribua os participantes **aleatoriamente** entre as variantes e não altere o teste durante a coleta.
4. Ao final, verifique o **balanceamento** dos grupos (SRM), aplique o teste e reporte a diferença com **IC**.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (confint_proportions_2indep, proportion_confint,
                                          proportion_effectsize, proportions_ztest)

# 1) Tamanho de amostra: quantos participantes por grupo para detectar o MDE?
taxa_base = 0.30          # taxa atual da métrica (histórico)
efeito_minimo = 0.03      # menor diferença absoluta relevante (MDE): de 30% para 33%

h = proportion_effectsize(taxa_base + efeito_minimo, taxa_base)
n_por_grupo = NormalIndPower().solve_power(effect_size=h, alpha=ALFA, power=0.8, ratio=1)
print(f'Amostra necessária: {np.ceil(n_por_grupo):,.0f} por grupo ({np.ceil(n_por_grupo) * 2:,.0f} no total)')
# Duração estimada = amostra total / participantes por dia

In [ ]:
# 2) Resultado do experimento, a partir da base de clientes (Seção 10.1)
ordem = ['tratamento', 'controle']
resumo_ab = (clientes.groupby('coluna_variante')['converteu']
                     .agg(n='count', conversoes='sum', taxa='mean')
                     .reindex(ordem))
ic_inf, ic_sup = proportion_confint(resumo_ab['conversoes'], resumo_ab['n'], alpha=ALFA, method='wilson')
resumo_ab['ic_inf'], resumo_ab['ic_sup'] = ic_inf, ic_sup
resumo_ab

In [ ]:
# 3) Balanceamento (SRM): os grupos têm o tamanho esperado (50/50)?
_, p_srm = stats.chisquare(resumo_ab['n'])
print(f'SRM: p = {p_srm:.4f} → {"grupos balanceados" if p_srm > 0.01 else "DESBALANCEADO: investigue antes de concluir"}')

# 4) Teste Z de duas proporções + IC da diferença
conv, n = resumo_ab['conversoes'].to_numpy(), resumo_ab['n'].to_numpy()
z, p_ab = proportions_ztest(count=conv, nobs=n)
dif_inf, dif_sup = confint_proportions_2indep(conv[0], n[0], conv[1], n[1], compare='diff', alpha=ALFA)

taxa_trat, taxa_ctrl = resumo_ab['taxa']
print(f'Taxas: tratamento = {taxa_trat:.2%} | controle = {taxa_ctrl:.2%}')
print(f'Diferença = {taxa_trat - taxa_ctrl:+.2%} (IC 95%: {dif_inf:+.2%} a {dif_sup:+.2%})')
print(f'Lift relativo = {taxa_trat / taxa_ctrl - 1:+.1%}')
print(f'Teste Z: z = {z:.2f}, p = {p_ab:.4g} → {"diferença significativa" if p_ab < ALFA else "sem evidência de diferença"}')

# 5) O experimento teve amostra suficiente? Sem poder, "sem diferença" pode ser só falta de dados
suficiente = n.min() >= n_por_grupo
print(f'Amostra por grupo: {n.min():,} de {np.ceil(n_por_grupo):,.0f} necessários → '
      + ('poder adequado para o MDE' if suficiente else 'teste SUBDIMENSIONADO: o resultado não descarta um efeito do tamanho do MDE'))

⚠️ **O que observar**
- **Não espie o resultado para parar antes** do tamanho de amostra planejado. Parar quando o p-valor "ficou bom" infla muito os falsos positivos.
- **SRM** (*sample ratio mismatch*): grupos com tamanhos muito diferentes do planejado indicam falha na aleatorização ou no registro. Nesse caso, o resultado não é confiável. O limiar usual é p < 0,01.
- **Resultado não significativo ≠ ausência de efeito.** Se a amostra ficou abaixo da planejada, ou se o IC da diferença ainda inclui o MDE, o teste é inconclusivo. Reporte assim.
- Reporte a **diferença absoluta** (pontos percentuais) com IC e o **lift relativo**. "+3 p.p." e "+10%" descrevem o mesmo efeito de formas diferentes.
- Para métricas **contínuas** (ticket médio, tempo), use o teste t de Welch ou Mann-Whitney (Seção 12.5). O tamanho de amostra vem de `TTestIndPower().solve_power(effect_size=d, alpha=ALFA, power=0.8)`, com `d` = d de Cohen esperado.
- Testar muitas métricas ou variantes ao mesmo tempo aumenta os falsos positivos. Defina **uma** métrica principal, ou corrija o α (Bonferroni: α / número de comparações).

## 12.9 Tamanho de efeito e intervalo de confiança

O **p-valor diz se existe diferença**. O **tamanho de efeito diz o quanto ela importa**. Com amostras grandes, diferenças irrelevantes ficam "significativas". Com amostras pequenas, diferenças relevantes podem não ficar. **Sempre reporte os dois.**

| Medida | Uso | Referência (Cohen) |
|---|---|---|
| **d de Cohen** = $\frac{\bar{x}_1 - \bar{x}_2}{s_{pooled}}$ | diferença entre 2 médias, em desvios padrão | 0,2 pequeno · 0,5 médio · 0,8 grande |
| **η² (eta quadrado)** = $\frac{SS_{entre}}{SS_{total}}$ | proporção da variância explicada pelos grupos (ANOVA) | 0,01 pequeno · 0,06 médio · 0,14 grande |
| **r rank-biserial** = $1 - \frac{2U}{n_1 n_2}$ | efeito do Mann-Whitney (-1 a 1) | 0,1 pequeno · 0,3 médio · 0,5 grande |
| **ε² (épsilon quadrado)** = $\frac{H}{n - 1}$ | efeito do Kruskal-Wallis (análogo ao η²) | mesmas referências do η² |
| **Intervalo de confiança (IC 95%)** | faixa plausível para a média (ou diferença) populacional | se o IC da diferença não contém 0, a diferença é significativa |

In [ ]:
def cohen_d(a, b):
    # d de Cohen com desvio padrão combinado (pooled).
    a, b = np.asarray(a), np.asarray(b)
    n1, n2 = len(a), len(b)
    s_pooled = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / s_pooled


def eta_quadrado(*amostras):
    # η² da ANOVA one-way: soma de quadrados entre grupos / soma de quadrados total.
    todos = np.concatenate(amostras)
    ss_total = ((todos - todos.mean()) ** 2).sum()
    ss_entre = sum(len(g) * (np.mean(g) - todos.mean()) ** 2 for g in amostras)
    return ss_entre / ss_total


def rank_biserial(a, b):
    # Tamanho de efeito do Mann-Whitney: proporção de pares favoráveis menos desfavoráveis.
    u, _ = stats.mannwhitneyu(a, b, alternative='two-sided')
    return 1 - 2 * u / (len(a) * len(b))


def epsilon_quadrado(*amostras):
    # Tamanho de efeito do Kruskal-Wallis.
    h, _ = stats.kruskal(*amostras)
    return h / (sum(len(g) for g in amostras) - 1)


def ic_media(valores, confianca=0.95):
    # Intervalo de confiança da média pela distribuição t.
    valores = np.asarray(valores)
    return stats.t.interval(confianca, df=len(valores) - 1, loc=valores.mean(), scale=stats.sem(valores))


print(f'd de Cohen (grupo_1 × grupo_2): {cohen_d(grupo_1, grupo_2):.2f}')
print(f'r rank-biserial               : {rank_biserial(grupo_1, grupo_2):.2f}')
print(f'η² (grupos de {var_cat})       : {eta_quadrado(*grupos.values()):.3f}')
print(f'ε² (grupos de {var_cat})       : {epsilon_quadrado(*grupos.values()):.3f}')
for nome, valores in grupos.items():
    li, ls = ic_media(valores)
    print(f'IC 95% da média do grupo {nome}: [{li:.2f}, {ls:.2f}]')

## 12.10 Fluxo completo automatizado

**O que faz:** aplica o roteiro da Seção 12.2 (pressupostos → escolha do teste → efeito) e devolve um resumo pronto para interpretar.

**Quando usar:** para comparações rápidas e padronizadas. Sempre olhe os gráficos e as tabelas de pressupostos antes de confiar na escolha automática.

In [ ]:
def comparar_grupos(data, var_num, var_cat, grupos_validos=None, alfa=ALFA):
    # Escolhe e aplica o teste adequado para comparar var_num entre os grupos de var_cat.
    dados = data.dropna(subset=[var_num, var_cat])
    if grupos_validos is not None:
        dados = dados[dados[var_cat].isin(grupos_validos)]
    amostras = {k: g[var_num].to_numpy() for k, g in dados.groupby(var_cat, observed=True)}
    vals = list(amostras.values())

    p_normal = [stats.shapiro(v)[1] for v in vals]
    normalidade = all(p > alfa for p in p_normal) or all(
        len(v) >= 30 and abs(stats.skew(v)) < 1 for v in vals)         # TLC: n grande e baixa assimetria
    homogeneidade = stats.levene(*vals, center='median')[1] > alfa

    if len(vals) == 2:
        if normalidade:
            teste = 't de Student' if homogeneidade else 't de Welch'
            estat, p = stats.ttest_ind(*vals, equal_var=homogeneidade)
            efeito = ('d de Cohen', cohen_d(*vals))
        else:
            teste, (estat, p) = 'Mann-Whitney', stats.mannwhitneyu(*vals, alternative='two-sided')
            efeito = ('r rank-biserial', rank_biserial(*vals))
    else:
        if normalidade:
            teste = 'ANOVA' if homogeneidade else 'ANOVA de Welch'
            estat, p = stats.f_oneway(*vals, equal_var=homogeneidade)
            efeito = ('η²', eta_quadrado(*vals))
        else:
            teste, (estat, p) = 'Kruskal-Wallis', stats.kruskal(*vals)
            efeito = ('ε²', epsilon_quadrado(*vals))

    return pd.Series({
        'grupos (n)': {k: len(v) for k, v in amostras.items()},
        'médias': {k: round(float(np.mean(v)), 2) for k, v in amostras.items()},
        'medianas': {k: round(float(np.median(v)), 2) for k, v in amostras.items()},
        'normalidade ok': normalidade,
        'variâncias homogêneas': homogeneidade,
        'teste': teste,
        'estatística': round(float(estat), 3),
        'p-valor': float(p),
        f'significativo (α={alfa})': p < alfa,
        'medida de efeito': efeito[0],
        'efeito': round(float(efeito[1]), 3),
    })

comparar_grupos(df, 'coluna_num', 'coluna_cat', grupos_validos=['A', 'B', 'C'])

In [ ]:
comparar_grupos(df, 'coluna_num_2', 'coluna_binaria')

## 12.11 🔹 Alternativa: `pingouin` (tabelas completas em uma linha)

**Quando usar:** em relatórios. Cada função devolve um DataFrame com a estatística, os graus de liberdade, o p-valor, o tamanho de efeito, o IC 95% e o poder, sem cálculos manuais.

| Etapa | Função |
|---|---|
| Normalidade por grupo | `pg.normality(data, dv=, group=)` |
| Homogeneidade (Levene) | `pg.homoscedasticity(data, dv=, group=)` |
| t (com Welch automático se variâncias diferentes) | `pg.ttest(x, y, correction='auto')` |
| ANOVA (com η² parcial) / ANOVA de Welch | `pg.anova(...)` / `pg.welch_anova(...)` |
| Pós-testes | `pg.pairwise_tukey(...)` / `pg.pairwise_gameshowell(..., effsize='cohen')` |
| Não paramétricos | `pg.mwu(x, y)`, `pg.kruskal(...)` |

In [ ]:
import pingouin as pg

pg.normality(dados_teste, dv=var_num, group=var_cat)

In [ ]:
pg.homoscedasticity(dados_teste, dv=var_num, group=var_cat)

In [ ]:
pg.ttest(grupo_1, grupo_2, correction='auto')     # inclui IC 95% da diferença e d de Cohen

In [ ]:
pg.welch_anova(data=dados_teste, dv=var_num, between=var_cat)

In [ ]:
pg.pairwise_gameshowell(data=dados_teste, dv=var_num, between=var_cat, effsize='cohen')

⚠️ O nome da coluna do p-valor varia entre versões do pingouin (`'p-val'` nas antigas, `'p_val'` nas recentes). Confira com `.columns` antes de acessar a coluna por nome.

---
# 13. Regressão linear (como ferramenta estatística)

**O que faz:** encontra a reta $y = a \cdot x + b$ que minimiza a distância dos pontos até ela (mínimos quadrados). Ela transforma a correlação em uma **regra quantitativa**: "a cada 1 unidade a mais em X, Y varia em média *a* unidades".

| Termo | Significado |
|---|---|
| `slope` (*a*, inclinação) | variação média de Y para cada unidade de X |
| `intercept` (*b*) | valor estimado de Y quando X = 0 (nem sempre tem sentido prático) |
| `rvalue` (r) | correlação de Pearson: o quanto a reta descreve bem os dados |
| R² = r² | proporção da variação de Y explicada por X (0 a 1) |
| `pvalue` | teste de H₀: inclinação = 0 (sem relação linear) |

**Quando usar:** para quantificar e descrever uma tendência linear entre duas variáveis numéricas. Aqui, o objetivo é **descrever e testar a relação**, não fazer previsões.

## 13.1 Regressão simples com `scipy`

In [ ]:
dados_reg = df[['coluna_num', 'coluna_num_2']].dropna()
reg = stats.linregress(dados_reg['coluna_num'], dados_reg['coluna_num_2'])

print(f'Reta: y = {reg.slope:.2f}·x + {reg.intercept:.2f}')
print(f'r = {reg.rvalue:.3f} | R² = {reg.rvalue**2:.3f} | p = {reg.pvalue:.3g} | erro padrão da inclinação = {reg.stderr:.3f}')

In [ ]:
def plot_regressao(data, x, y, titulo=''):
    # Dispersão + reta de regressão calculada pelo scipy, com equação e R² no título.
    d = data[[x, y]].dropna()
    r = stats.linregress(d[x], d[y])
    xs = np.linspace(d[x].min(), d[x].max(), 100)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.scatterplot(data=d, x=x, y=y, alpha=0.4, ax=ax)
    ax.plot(xs, r.slope * xs + r.intercept, color='#efab02', linewidth=2.5)
    ax.set_title(f'{titulo} y = {r.slope:.2f}x + {r.intercept:.2f} | r = {r.rvalue:.2f} | R² = {r.rvalue**2:.2f}')
    plt.show()
    return r

plot_regressao(df, 'coluna_num', 'coluna_num_2', 'Com outliers:');

## 13.2 Impacto dos outliers

Poucos pontos extremos "puxam" a reta e mudam a inclinação e o r. Compare sempre os resultados com e sem eles (os critérios estão na Seção 7).

In [ ]:
plot_regressao(df_sem_outliers, 'coluna_num', 'coluna_num_2', 'Sem outliers:');

## 13.3 Regressão visual com `seaborn`

**O que faz:** `regplot` (axes-level) e `lmplot` (figure-level, com `hue`/`col`/`row`) desenham a reta e a **faixa de confiança** em uma linha de código.

⚠️ Esses gráficos **não mostram** a inclinação, o intercepto ou o r. Quando os números importam, calcule-os com `scipy` ou `statsmodels`.

In [ ]:
fig, axes = plt.subplots(ncols=3, figsize=(18, 4), sharey=True)
sns.regplot(data=df_sem_outliers, x='coluna_num', y='coluna_num_2', ax=axes[0],
            scatter_kws={'alpha': 0.3, 's': 10}, line_kws={'color': 'red'})
axes[0].set_title('Linear (faixa = IC 95%)')
sns.regplot(data=df_sem_outliers, x='coluna_num', y='coluna_num_2', order=2, ax=axes[1],
            scatter_kws={'alpha': 0.3, 's': 10})
axes[1].set_title('Polinomial (order=2)')
sns.regplot(data=df_sem_outliers, x='coluna_num', y='coluna_num_2', logx=True, ax=axes[2],
            scatter_kws={'alpha': 0.3, 's': 10})
axes[2].set_title('y ~ log(x) (logx=True; exige x > 0)')
plt.show()

In [ ]:
# Uma reta por grupo (hue) e/ou um painel por categoria (col/row)
sns.lmplot(data=df_sem_outliers, x='coluna_num', y='coluna_num_2', hue='coluna_cat_2', col='coluna_cat',
           col_order=['A', 'B', 'C'], markers=['o', 'v'], scatter_kws={'s': 15, 'alpha': 0.4},
           height=4, aspect=1)
plt.show()

🔹 Outras opções úteis:
- `sns.residplot(data=..., x=..., y=...)` ➕ mostra os resíduos. Eles devem formar uma nuvem sem padrão em torno de zero; uma curva ou um "funil" indicam relação não linear ou variância não constante.
- `regplot(logistic=True)`, quando Y é binário (0/1). Requer `statsmodels`.

## 13.4 Regressão com `statsmodels` (OLS) ➕

**O que faz:** retorna a tabela estatística completa: coeficientes com **IC 95%** e p-valor, R² ajustado, e permite **vários preditores** e variáveis categóricas.

**Quando usar:** quando a regressão entra no relatório, ou para medir o efeito de uma variável **controlando** outras.

In [ ]:
import statsmodels.formula.api as smf

modelo = smf.ols('coluna_num_2 ~ coluna_num + C(coluna_cat_2)', data=df_sem_outliers).fit()
print(modelo.summary().tables[1])             # coeficientes, erro padrão, p-valor, IC 95%
print(f'\nR² = {modelo.rsquared:.3f} | R² ajustado = {modelo.rsquared_adj:.3f}')

⚠️ **O que observar**
- **Pressupostos**: relação linear, resíduos independentes, variância constante dos resíduos e resíduos aproximadamente normais (para os p-valores). Verifique-os com `residplot` e com um QQ-plot dos resíduos (`stats.probplot(modelo.resid, plot=ax)`).
- **R² baixo não invalida a relação**: a inclinação pode ser significativa, mas X explica pouco da variação de Y. Reporte os dois.
- Não extrapole a reta para fora do intervalo observado de X.
- Regressão mostra **associação**, não causalidade (mesmo cuidado da correlação).
- Em `C(coluna_cat)`, cada coeficiente é a diferença em relação à **categoria de referência** (a primeira em ordem alfabética).

---
# 14. Insights e comunicação

## 14.1 Como extrair insights

Um insight é um **padrão relevante para a decisão**, com evidência e com uma explicação plausível. Para cada resultado, pergunte:

1. **O quê?** Qual é o padrão, com números (n, %, média, diferença)?
2. **É real?** Ele se sustenta com o teste estatístico, o tamanho de efeito, o n suficiente e sem depender de outliers?
3. **Por quê?** Qual é a explicação mais plausível? Há explicações alternativas (terceira variável, viés de coleta)?
4. **E daí?** Que decisão ou ação ele sugere?

**Padrões que costumam gerar insights**

| Sinal nos dados | Possível interpretação | Onde aparece |
|---|---|---|
| Picos em valores redondos (5, 10, 15...) | dados declarados e arredondados por quem respondeu | histograma (6.1) |
| Média bem maior que a mediana | poucos valores altos dominam: reportar a mediana | `describe` (5.1) |
| Duas "nuvens" na dispersão | duas populações misturadas: segmentar | dispersão (8.5) |
| Correlação geral some dentro dos grupos | a relação é explicada pelo grupo, não pela variável | correlação por grupo (8.5) |
| Diferença entre grupos ligada à "idade" do registro | efeito de geração/coorte (quem entrou antes tem perfil diferente) | `groupby` + violino (8.3) |
| Muitos nulos em um perfil específico | ausência não aleatória: conclusões enviesadas | nulos (3.3) |
| Poucos casos extremos com grande peso no total | concentração (regra 80/20): priorizar esses casos | outliers (7.3), ranking (8.3) |
| Diferença significativa, mas efeito pequeno | estatisticamente real, mas pouco relevante na prática | tamanho de efeito (12.9) |

## 14.2 Como reportar resultados estatísticos

Use sempre o mesmo padrão, com **estatística, graus de liberdade, p-valor, tamanho de efeito e n**:

| Resultado | Modelo de frase |
|---|---|
| Descritiva | "O grupo A apresentou média de **X ± DP** (mediana **Y**; n = **N**)." |
| Teste t | "O grupo A apresentou média superior ao grupo B (**t**(gl) = valor; **p** < 0,001; **d** = valor; IC 95% da diferença: [li; ls])." |
| ANOVA | "Houve diferença entre os grupos (**F**(gl₁, gl₂) = valor; **p** < 0,001; **η²** = valor). O pós-teste de Games-Howell indicou que..." |
| Não paramétrico | "As distribuições diferiram (Mann-Whitney **U** = valor; **p** = valor; **r** = valor). Medianas: A = ..., B = ..." |
| Correlação | "Houve correlação positiva moderada entre X e Y (Spearman **ρ** = valor; **p** < 0,001; n = N)." |
| Pressupostos | "Normalidade avaliada por Shapiro-Wilk e homogeneidade por Levene (α = 0,05). Por violação da homocedasticidade, usou-se o teste t com correção de Welch." |

In [ ]:
def formatar_p(p):
    # Formata o p-valor no padrão de relatório.
    return 'p < 0,001' if p < 0.001 else f'p = {p:.3f}'.replace('.', ',')

res = comparar_grupos(df, 'coluna_num_2', 'coluna_binaria')
print(f"{res['teste']}: estatística = {res['estatística']}; {formatar_p(res['p-valor'])}; "
      f"{res['medida de efeito']} = {res['efeito']}")

## 14.3 Gráfico final com significância anotada

**O que faz:** mostra as médias dos grupos com barras de erro, os valores nas barras e a **marcação de significância** entre os pares comparados.

**Quando usar:** no relatório, para comunicar o resultado de um teste de forma visual.

In [ ]:
def anotar_significancia(ax, x1, x2, y, p, altura=0.02):
    # Desenha um colchete entre as posições x1 e x2 com o p-valor (ou asteriscos).
    estrelas = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    h = altura * (ax.get_ylim()[1] - ax.get_ylim()[0])
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='black', linewidth=1.2)
    ax.text((x1 + x2) / 2, y + h, f'{estrelas}\n{formatar_p(p)}', ha='center', va='bottom', fontsize=10)


ordem = [0, 1]
fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(data=df, x='coluna_binaria', y='coluna_num_2', hue='coluna_binaria', order=ordem,
            errorbar='sd', capsize=0.1, palette='viridis', width=0.5, edgecolor='black', legend=False, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', label_type='center', color='white', fontsize=13)

_, p_par = stats.ttest_ind(grupo_1, grupo_2, equal_var=False)
y_max = df.groupby('coluna_binaria')['coluna_num_2'].agg(lambda s: s.mean() + s.std()).max()
ax.set_ylim(0, y_max * 1.3)
anotar_significancia(ax, 0, 1, y_max * 1.05, p_par)

ax.set_xticks([0, 1], ['Grupo 0', 'Grupo 1'])
ax.set_xlabel('')
ax.set_ylabel('Média de coluna_num_2 (± DP)')
plt.tight_layout()
plt.show()

🔹 **Alternativa: biblioteca `statannotations`** (`pip install statannotations`). Ela anota pares automaticamente a partir de p-valores já calculados, por exemplo os de um pós-teste:

```python
from statannotations.Annotator import Annotator

pares = list(zip(pos_teste['A'], pos_teste['B']))          # pares do pós-teste (ex.: pg.pairwise_gameshowell)
annotator = Annotator(ax, pares, data=df, x='coluna_cat', y='coluna_num')
annotator.configure(test=None, text_format='simple', loc='inside')
annotator.set_pvalues_and_annotate(pos_teste['pval'].values)
```
⚠️ Verifique a compatibilidade da versão do `statannotations` com o seu `seaborn`.

## 14.4 Estrutura do relatório

Comece pelo que o leitor precisa decidir e deixe o detalhe técnico para o final.

```text
[TÍTULO DA ANÁLISE]
Responsável: [nome] | Data: [dd/mm/aaaa] | Fonte dos dados: [origem, período]

1. RESUMO EXECUTIVO (máx. 1 página)
   - Contexto: o que foi analisado (n, período, objetivo)
   - O que encontramos: 2-3 achados principais, em linguagem de negócio
   - Por que isso importa: impacto para a decisão
   - Ações prioritárias: 2-3 recomendações

2. PRINCIPAIS ACHADOS
   [Título do achado em forma de conclusão (ex.: "Grupo X apresenta desempenho inferior")]
   - Evidência numérica: médias, %, n por grupo, diferença e IC
   - Gráfico de suporte (1 por achado)

3. IMPLICAÇÕES
   - O que cada achado significa para o negócio (indicador de risco, alavanca, oportunidade)

4. AÇÕES SUGERIDAS
   - Curto prazo: ações rápidas, com critério objetivo (ex.: "monitorar casos com métrica < limite")
   - Médio/longo prazo: mudanças estruturais, novos indicadores, integração a sistemas

5. DETALHAMENTO TÉCNICO
   5.1 Dados e tratamento: filtros, remoções (quantas linhas e por quê), transformações
   5.2 Análises descritivas: métricas usadas (média ± DP, CV, assimetria, curtose)
   5.3 Análises inferenciais: pressupostos (Shapiro-Wilk, Levene), testes escolhidos e motivo,
       α, estatísticas, p-valores, tamanhos de efeito, pós-testes
   5.4 Limitações: viés de amostra, nulos, causalidade não comprovada
```

⚠️ **O que observar**
- Títulos de achados devem ser **conclusões**, não descrições ("O grupo A converte 2x mais", e não "Conversão por grupo").
- Uma ideia por gráfico, e um gráfico por achado.
- Diga explicitamente quando o resultado é **associação**, e não causa.

## 14.5 Exportação e pipeline reprodutível

**O que faz:** salva as tabelas e gráficos finais e reúne o tratamento em um bloco único que pode ser executado de novo com novos dados.

**Quando usar:** ao final da exploração, quando as decisões de limpeza e as métricas já estão definidas. Leve para um bloco único (ou um script `.py`) **apenas** o que entra no resultado final.

In [ ]:
# Tabela final: consolidada, filtrada e ordenada
tabela_final = (df
    .groupby('coluna_cat')
    .agg(n=('coluna_id', 'count'), total=('coluna_valor', 'sum'),
         media=('coluna_num_2', 'mean'), duracao_media_dias=('duracao_dias', 'mean'))
    .query('n >= 30')                                   # só grupos com n mínimo
    .sort_values('total', ascending=False)
    .round(2)
    .reset_index())

PASTA_SAIDA.mkdir(exist_ok=True)
tabela_final.to_excel(CAMINHO_SAIDA_TABELA, index=False)          # requer openpyxl
tabela_final.to_csv(PASTA_SAIDA / f'tabela_{PERIODO}.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
tabela_final

**Modelo de pipeline em bloco único** (parâmetros no topo → leitura → tratamento → consolidação → exportação):

```python
# ===== Parâmetros =====
PERIODO = 'aaaa'
CAMINHO_DADOS = Path('dados') / f'{PERIODO}_nome_do_arquivo.csv'
CAMINHO_SAIDA = Path('output')

# ===== Leitura =====
df = pd.read_csv(CAMINHO_DADOS, sep=';', encoding='cp1252', decimal=',', thousands='.')

# ===== Tratamento =====
df['coluna_cat'] = df['coluna_cat'].fillna('Não informado')
df['coluna_data'] = pd.to_datetime(df['coluna_data'], format='%d/%m/%Y')
df['coluna_total'] = df[['coluna_1', 'coluna_2', 'coluna_3']].sum(axis=1)

# ===== Consolidação =====
tabela = (df.groupby('coluna_cat')
            .agg(n=('coluna_id', 'count'), total=('coluna_total', 'sum'))
            .sort_values('total', ascending=False)
            .reset_index())

# ===== Exportação =====
tabela.to_excel(CAMINHO_SAIDA / f'tabela_{PERIODO}.xlsx', index=False)
# + gráfico final com fig.savefig(CAMINHO_SAIDA / f'grafico_{PERIODO}.png', bbox_inches='tight')
```

---
# 15. Checklist final

**Antes de começar**
- [ ] A pergunta de negócio está clara? O que significa cada linha da tabela?
- [ ] Li o dicionário de dados (ou documentei as colunas, unidades e a fonte)?

**Dados**
- [ ] O arquivo foi lido corretamente (`sep`, `encoding`, `decimal`) e o `shape` está coerente?
- [ ] Os tipos estão corretos (números, datas, categorias ordenadas)?
- [ ] Nulos: quantifiquei, entendi o motivo e decidi o tratamento?
- [ ] Duplicados: verifiquei pela linha e pela chave?
- [ ] Categorias: padronizei os rótulos e tratei as categorias não informativas ou raras?
- [ ] Registrei quantas linhas cada filtro ou remoção eliminou?
- [ ] Junções: validei a cardinalidade da chave e o número de linhas antes e depois do `merge`?
- [ ] Consultas SQL: filtrei e agreguei no banco antes de trazer os dados, usando parâmetros em vez de concatenar texto?

**Exploração**
- [ ] Fiz a descritiva completa (média, mediana, DP, CV, quartis, assimetria, curtose), geral e por segmento?
- [ ] Olhei o histograma ou boxplot de cada variável importante?
- [ ] Investiguei os outliers (erro × caso legítimo) e medi o impacto deles?
- [ ] Explorei as relações (correlação, comparação entre grupos, tabelas cruzadas, tempo)?
- [ ] Verifiquei se a relação muda entre segmentos?
- [ ] Comparei períodos (mês a mês, ano a ano, acumulado) usando períodos completos e equivalentes?
- [ ] Defini cada KPI com numerador, denominador, período e unidade, e analisei funil e retenção quando aplicável?

**Inferência**
- [ ] Defini H₀, H₁ e α antes de testar?
- [ ] Verifiquei os pressupostos (independência, normalidade, homogeneidade)?
- [ ] Usei o teste adequado (paramétrico × não paramétrico; 2 × 3+ grupos; pareado × independente)?
- [ ] Apliquei o pós-teste quando havia 3+ grupos?
- [ ] Reportei o tamanho de efeito e o IC, e não só o p-valor?
- [ ] Testes A/B: calculei o tamanho de amostra antes, verifiquei o balanceamento dos grupos (SRM) e reportei o lift com IC?

**Comunicação**
- [ ] Cada achado tem número, evidência, interpretação e ação?
- [ ] Os gráficos têm título, rótulos, unidade e fonte, e a ordem e as cores são consistentes?
- [ ] Deixei claras as limitações (amostra, nulos, causalidade)?
- [ ] As tabelas e os gráficos finais foram exportados, e o pipeline pode ser executado de novo?